In [ ]:
import sys
print(f"Python version: {sys.version}")
print("Basic import successful!")

## 1. Import Required Libraries

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


import sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, chi2, RFE, mutual_info_classif
from sklearn.decomposition import PCA


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier


try:
    import xgboost as xgb
    print("✓ XGBoost imported successfully")
except Exception as e:
    print(f"⚠ XGBoost not available. Run: brew install libomp")
    xgb = None

try:
    import lightgbm as lgb
    print("✓ LightGBM imported successfully")
except Exception as e:
    print(f"⚠ LightGBM not available: {str(e)[:50]}...")
    lgb = None

try:
    from catboost import CatBoostClassifier
    print("✓ CatBoost imported successfully")
except Exception as e:
    print(f"⚠ CatBoost not available: {str(e)[:50]}...")
    CatBoostClassifier = None


from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    confusion_matrix, classification_report, matthews_corrcoef
)


from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek


try:
    import shap
    print("✓ SHAP imported successfully")
except Exception as e:
    print(f"⚠ SHAP not available: {str(e)[:50]}...")
    shap = None

try:
    from lime import lime_tabular
    print("✓ LIME imported successfully")
except Exception as e:
    print(f"⚠ LIME not available: {str(e)[:50]}...")
    lime_tabular = None


import joblib
import pickle

from scipy import stats
from scipy.stats import zscore


from datetime import datetime


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.4f' % x)


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print("✓ Core libraries imported successfully!")

print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")


## 2. Define File Paths and Configuration

In [ ]:

BASE_DIR = Path('/Users/roya/Desktop/Thesis')
DATA_DIR = BASE_DIR / 'Data set'
OUTPUT_DIR = BASE_DIR / 'Thesis_Project' / 'ML'


PART_B_FILE = DATA_DIR / 'Part B' / '2023' / 'MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv'
PART_D_FILE = DATA_DIR / 'Medicare Part D Prescribers - by Provider and Drug' / '2023' / 'MUP_DPR_RY25_P04_V10_DY23_NPIBN.csv'
OIG_FILE = DATA_DIR / 'OIG.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'models').mkdir(exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(exist_ok=True)
(OUTPUT_DIR / 'reports').mkdir(exist_ok=True)

print("✓ File paths configured")
print(f"Base Directory: {BASE_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

## 3. Data Loading & Initial Exploration

### 3.1 Load Medicare Part B Data

In [ ]:
print("Loading Medicare Part B 2023 data...")
part_b_df = pd.read_csv(PART_B_FILE, low_memory=False)

print(f"Part B Data Shape: {part_b_df.shape}")
print(f"\nMemory Usage: {part_b_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
part_b_df.head()

### 3.2

In [ ]:
us_states = ['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 
             'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 
             'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ', 
             'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 
             'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 
             'DC', 'PR', 'VI', 'GU', 'AS', 'MP']

before_filter_b = len(part_b_df)
part_b_df = part_b_df[part_b_df['Rndrng_Prvdr_State_Abrvtn'].isin(us_states)]
after_filter_b = len(part_b_df)
removed_b = before_filter_b - after_filter_b

print(f"Part B: Removed {removed_b:,} non-U.S. records ({removed_b/before_filter_b*100:.2f}%)")
print(f"Part B: Remaining records: {after_filter_b:,}")
print(f"Part B Data Shape after filtering: {part_b_df.shape}")

### 3.3

In [ ]:

print("PROVIDER LOCATION CONSISTENCY CHECK - PART B")

provider_location_check_b = part_b_df.groupby('Rndrng_NPI').agg({
    'Rndrng_Prvdr_State_Abrvtn': 'nunique',
    'Rndrng_Prvdr_Zip5': 'nunique',
    'Rndrng_Prvdr_RUCA': 'nunique'
}).rename(columns={
    'Rndrng_Prvdr_State_Abrvtn': 'Unique_States',
    'Rndrng_Prvdr_Zip5': 'Unique_Zipcodes',
    'Rndrng_Prvdr_RUCA': 'Unique_RUCA'
})

total_providers = len(provider_location_check_b)
single_state = (provider_location_check_b['Unique_States'] == 1).sum()
multiple_states = (provider_location_check_b['Unique_States'] > 1).sum()

single_zip = (provider_location_check_b['Unique_Zipcodes'] == 1).sum()
multiple_zip = (provider_location_check_b['Unique_Zipcodes'] > 1).sum()

single_ruca = (provider_location_check_b['Unique_RUCA'] == 1).sum()
multiple_ruca = (provider_location_check_b['Unique_RUCA'] > 1).sum()

print(f"\nTotal unique providers: {total_providers:,}")
print(f"\n{'Field':<20} {'Single Value':<20} {'Multiple Values':<20}")

print(f"{'State':<20} {single_state:,} ({single_state/total_providers*100:.2f}%)    {multiple_states:,} ({multiple_states/total_providers*100:.2f}%)")
print(f"{'Zipcode':<20} {single_zip:,} ({single_zip/total_providers*100:.2f}%)    {multiple_zip:,} ({multiple_zip/total_providers*100:.2f}%)")
print(f"{'RUCA Code':<20} {single_ruca:,} ({single_ruca/total_providers*100:.2f}%)    {multiple_ruca:,} ({multiple_ruca/total_providers*100:.2f}%)")

print("MIN AND MAX STATISTICS FOR UNIQUE VALUES")

print(f"\n{'Field':<20} {'Min':<15} {'Max':<15} {'Mean':<15}")

print(f"{'Unique_States':<20} {provider_location_check_b['Unique_States'].min():<15} {provider_location_check_b['Unique_States'].max():<15} {provider_location_check_b['Unique_States'].mean():<15.2f}")
print(f"{'Unique_Zipcodes':<20} {provider_location_check_b['Unique_Zipcodes'].min():<15} {provider_location_check_b['Unique_Zipcodes'].max():<15} {provider_location_check_b['Unique_Zipcodes'].mean():<15.2f}")
print(f"{'Unique_RUCA':<20} {provider_location_check_b['Unique_RUCA'].min():<15} {provider_location_check_b['Unique_RUCA'].max():<15} {provider_location_check_b['Unique_RUCA'].mean():<15.2f}")

print("DETAILED PROVIDER LOCATION TABLE")



provider_location_check_b['Has_Multiple_Locations'] = (
    (provider_location_check_b['Unique_States'] > 1) |
    (provider_location_check_b['Unique_Zipcodes'] > 1) |
    (provider_location_check_b['Unique_RUCA'] > 1)
)


print(f"\nShowing all {len(provider_location_check_b):,} providers:")
display(provider_location_check_b)

if multiple_states > 0 or multiple_zip > 0 or multiple_ruca > 0:

    print("PROVIDERS WITH INCONSISTENT LOCATIONS")

    multi_location_providers = provider_location_check_b[
        provider_location_check_b['Has_Multiple_Locations']
    ]
    print(f"\nTotal providers with multiple locations: {len(multi_location_providers):,}")
    display(multi_location_providers)


In [ ]:
print(part_b_df.columns)

remove = ['Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_First_Name', 'Rndrng_Prvdr_MI', 'Rndrng_Prvdr_Crdntls', 'Rndrng_Prvdr_St1', 'Rndrng_Prvdr_St2', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_RUCA_Desc', 'Rndrng_Prvdr_Cntry', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'HCPCS_Desc', ]

keep = [ 'Rndrng_NPI', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_Zip5', Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_Cntry', 'Rndrng_Prvdr_Ent_Cd', '', 'Rndrng_Prvdr_Type', 'HCPCS_Cd', 'HCPCS_Drug_Ind', 'Place_Of_Srvc', 'Tot_Benes',	'Tot_Srvcs', 'Tot_Bene_Day_Srvcs', 'Avg_Sbmtd_Chrg',	'Avg_Mdcr_Alowd_Amt', 	'Avg_Mdcr_Pymt_Amt', 	'Avg_Mdcr_Stdzd_Amt'   ]

In [ ]:
part_b_df['Rndrng_Prvdr_Ent_Cd'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_Cntry'].unique()

In [ ]:
part_b_df['HCPCS_Drug_Ind'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_Crdntls'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_Mdcr_Prtcptg_Ind'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_RUCA'].unique()

### 3.4

In [ ]:
provider_summary = part_b_df.groupby('Rndrng_NPI').agg({
    'Rndrng_Prvdr_Type': lambda x: list(x.unique()),
    'HCPCS_Cd': lambda x: list(x.unique()),
    'HCPCS_Drug_Ind': lambda x: list(x.unique())
}).reset_index()

provider_summary['Num_Unique_Provider_Types'] = provider_summary['Rndrng_Prvdr_Type'].apply(len)
provider_summary['Num_Unique_HCPCS_Codes'] = provider_summary['HCPCS_Cd'].apply(len)
provider_summary['Num_Unique_HCPCS_Drug_Ind'] = provider_summary['HCPCS_Drug_Ind'].apply(len)

provider_summary['Provider_Types'] = provider_summary['Rndrng_Prvdr_Type'].apply(lambda x: ', '.join(map(str, x)))
provider_summary['HCPCS_Codes'] = provider_summary['HCPCS_Cd'].apply(lambda x: ', '.join(map(str, x)))
provider_summary['HCPCS_Drug_Ind_List'] = provider_summary['HCPCS_Drug_Ind'].apply(lambda x: ', '.join(map(str, x)))

provider_summary_display = provider_summary[[
    'Rndrng_NPI',
    'Num_Unique_Provider_Types',
    'Num_Unique_HCPCS_Codes',
    'Num_Unique_HCPCS_Drug_Ind',
    'Provider_Types',
    'HCPCS_Codes',
    'HCPCS_Drug_Ind_List'
]]

print(f"\nTotal number of unique providers: {len(provider_summary_display)}")
print(f"\nFirst 20 providers:")
print(provider_summary_display.head(20).to_string(index=False))


print("SUMMARY STATISTICS")
print(f"Average provider types per provider: {provider_summary_display['Num_Unique_Provider_Types'].mean():.2f}")
print(f"Average HCPCS codes per provider: {provider_summary_display['Num_Unique_HCPCS_Codes'].mean():.2f}")
print(f"Average HCPCS Drug Ind per provider: {provider_summary_display['Num_Unique_HCPCS_Drug_Ind'].mean():.2f}")
print(f"Max provider types for a single provider: {provider_summary_display['Num_Unique_Provider_Types'].max()}")
print(f"Max HCPCS codes for a single provider: {provider_summary_display['Num_Unique_HCPCS_Codes'].max()}")
print(f"Max HCPCS Drug Ind for a single provider: {provider_summary_display['Num_Unique_HCPCS_Drug_Ind'].max()}")


provider_summary_display

In [ ]:
part_b_df['Place_Of_Srvc'].unique()

In [ ]:
part_b_df['HCPCS_Drug_Ind'].unique()

In [ ]:
print("Part B Data Info:")
part_b_df.info()

In [ ]:
part_b_df.describe()

### 3.2 Load Medicare Part D Data

### 3.2.1

In [ ]:
print("Loading Medicare Part D 2023 data...")
part_d_df = pd.read_csv(PART_D_FILE, low_memory=False)

print(f"Part D Data Shape: {part_d_df.shape}")
print(f"\nMemory Usage: {part_d_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

part_d_df.head()

### 3.2.2

In [ ]:
before_filter_d = len(part_d_df)
part_d_df = part_d_df[part_d_df['Prscrbr_State_Abrvtn'].isin(us_states)]
after_filter_d = len(part_d_df)
removed_d = before_filter_d - after_filter_d

print(f"Part D: Removed {removed_d:,} non-U.S. records ({removed_d/before_filter_d*100:.2f}%)")
print(f"Part D: Remaining records: {after_filter_d:,}")
print(f"Part D Data Shape after filtering: {part_d_df.shape}")

### 3.2.3

In [ ]:
print("PROVIDER LOCATION CONSISTENCY CHECK - PART D")
provider_location_check_d = part_d_df.groupby('Prscrbr_NPI').agg({
    'Prscrbr_State_Abrvtn': 'nunique',
    'Prscrbr_State_FIPS': 'nunique'
}).rename(columns={
    'Prscrbr_State_Abrvtn': 'Unique_States',
    'Prscrbr_State_FIPS': 'Unique_State_FIPS'
})


total_providers_d = len(provider_location_check_d)
single_state_d = (provider_location_check_d['Unique_States'] == 1).sum()
multiple_states_d = (provider_location_check_d['Unique_States'] > 1).sum()

single_fips_d = (provider_location_check_d['Unique_State_FIPS'] == 1).sum()
multiple_fips_d = (provider_location_check_d['Unique_State_FIPS'] > 1).sum()

print(f"\nTotal unique prescribers: {total_providers_d:,}")
print(f"\n{'Field':<20} {'Single Value':<20} {'Multiple Values':<20}")

print(f"{'State':<20} {single_state_d:,} ({single_state_d/total_providers_d*100:.2f}%)    {multiple_states_d:,} ({multiple_states_d/total_providers_d*100:.2f}%)")
print(f"{'State FIPS':<20} {single_fips_d:,} ({single_fips_d/total_providers_d*100:.2f}%)    {multiple_fips_d:,} ({multiple_fips_d/total_providers_d*100:.2f}%)")

print("MIN AND MAX STATISTICS FOR UNIQUE VALUES")

print(f"\n{'Field':<20} {'Min':<15} {'Max':<15} {'Mean':<15}")

print(f"{'Unique_States':<20} {provider_location_check_d['Unique_States'].min():<15} {provider_location_check_d['Unique_States'].max():<15} {provider_location_check_d['Unique_States'].mean():<15.2f}")
print(f"{'Unique_State_FIPS':<20} {provider_location_check_d['Unique_State_FIPS'].min():<15} {provider_location_check_d['Unique_State_FIPS'].max():<15} {provider_location_check_d['Unique_State_FIPS'].mean():<15.2f}")

print("DETAILED PROVIDER LOCATION TABLE")


provider_location_check_d['Has_Multiple_Locations'] = (
    (provider_location_check_d['Unique_States'] > 1) |
    (provider_location_check_d['Unique_State_FIPS'] > 1)
)


print(f"\nShowing all {len(provider_location_check_d):,} prescribers:")
display(provider_location_check_d)


if multiple_states_d > 0 or multiple_fips_d > 0:

    print("PRESCRIBERS WITH INCONSISTENT LOCATIONS")
  
    multi_location_prescribers = provider_location_check_d[
        provider_location_check_d['Has_Multiple_Locations']
    ]
    print(f"\nTotal prescribers with multiple locations: {len(multi_location_prescribers):,}")
    display(multi_location_prescribers)



In [ ]:
print(part_d_df.columns)

remove = ['Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_City', 'Prscrbr_State_FIPS', 'Prscrbr_Type_Src',  'GE65_Sprsn_Flag', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Tot_Benes']

keep = ['Prscrbr_NPI', , 'Prscrbr_State_Abrvtn', 'Prscrbr_Type', 'Brnd_Name', 'Gnrc_Name','Tot_Clms', 'Tot_30day_Fills', 'Tot_Day_Suply', 'Tot_Drug_Cst','Tot_Benes']

Tot_30day_Fills': 'sum',  # How many monthly prescriptions the provider wrote

'Tot_Day_Suply': 'sum',  # Total number of days of medication given to patients

'Tot_Benes': 'sum',  # Number of different Medicare patients who received this drug from this provider(note: 0 means suppressed <11)
    
 how much money
'Tot_Drug_Cst': ['sum'],  # total money spent at pharmacies because of this provider’s prescriptions

### 3.2.4

In [ ]:
part_d_provider_summary = part_d_df.groupby('Prscrbr_NPI').agg({
    'Brnd_Name': lambda x: ', '.join(sorted(x.unique())),  
    'Gnrc_Name': lambda x: ', '.join(sorted(x.unique())),  
    'Tot_Clms': 'sum',          
    'Tot_30day_Fills': 'sum',    
    'Tot_Day_Suply': 'sum',      
    'Tot_Drug_Cst': 'sum',      
    'Tot_Benes': 'sum'          
}).reset_index()


part_d_provider_summary['Num_Unique_Brand_Names'] = part_d_df.groupby('Prscrbr_NPI')['Brnd_Name'].nunique().values
part_d_provider_summary['Num_Unique_Generic_Names'] = part_d_df.groupby('Prscrbr_NPI')['Gnrc_Name'].nunique().values

part_d_provider_summary = part_d_provider_summary[[
    'Prscrbr_NPI',
    'Tot_Clms',
    'Tot_30day_Fills',
    'Tot_Day_Suply',
    'Tot_Drug_Cst',
    'Tot_Benes',
    'Num_Unique_Brand_Names',
    'Num_Unique_Generic_Names',
    'Brnd_Name',
    'Gnrc_Name'
]]

print(f"\nTotal unique providers: {len(part_d_provider_summary):,}")
print(f"\nSummary statistics:")
print(f"  Average claims per provider: {part_d_provider_summary['Tot_Clms'].mean():,.2f}")
print(f"  Average drug cost per provider: ${part_d_provider_summary['Tot_Drug_Cst'].mean():,.2f}")
print(f"  Average unique brand names per provider: {part_d_provider_summary['Num_Unique_Brand_Names'].mean():.2f}")
print(f"  Average unique generic names per provider: {part_d_provider_summary['Num_Unique_Generic_Names'].mean():.2f}")

print(f"\nFirst 10 providers:")

part_d_provider_summary.head(10)

### 3.2.5

In [ ]:
print("PROVIDER SUMMARY TABLE - PART D")

provider_summary_d = part_d_df.groupby('Prscrbr_NPI').agg({
    'Prscrbr_Type': lambda x: list(x.unique()), 
    'Brnd_Name': lambda x: list(x.unique()),
    'Gnrc_Name': lambda x: list(x.unique())
}).reset_index() 


provider_summary_d['Num_Unique_Provider_Types'] = provider_summary_d['Prscrbr_Type'].apply(len)
provider_summary_d['Num_Unique_Brnd_Names'] = provider_summary_d['Brnd_Name'].apply(len)
provider_summary_d['Num_Unique_Gnrc_Names'] = provider_summary_d['Gnrc_Name'].apply(len)


provider_summary_d['Provider_Types'] = provider_summary_d['Prscrbr_Type'].apply(lambda x: ', '.join(map(str, x)))
provider_summary_d['Brand_Names'] = provider_summary_d['Brnd_Name'].apply(lambda x: ', '.join(map(str, x)))
provider_summary_d['Generic_Names'] = provider_summary_d['Gnrc_Name'].apply(lambda x: ', '.join(map(str, x)))


provider_summary_d_display = provider_summary_d[[
    'Prscrbr_NPI',
    'Num_Unique_Provider_Types',
    'Num_Unique_Brnd_Names',
    'Num_Unique_Gnrc_Names',
    'Provider_Types',
    'Brand_Names',
    'Generic_Names'
]]

print(f"\nTotal number of unique providers: {len(provider_summary_d_display)}")
print(f"\nFirst 20 providers:")
print(provider_summary_d_display.head(20).to_string(index=False))

print("SUMMARY STATISTICS - PART D")
print(f"Average provider types per provider: {provider_summary_d_display['Num_Unique_Provider_Types'].mean():.2f}")
print(f"Average brand names per provider: {provider_summary_d_display['Num_Unique_Brnd_Names'].mean():.2f}")
print(f"Average generic names per provider: {provider_summary_d_display['Num_Unique_Gnrc_Names'].mean():.2f}")
print(f"Max provider types for a single provider: {provider_summary_d_display['Num_Unique_Provider_Types'].max()}")
print(f"Max brand names for a single provider: {provider_summary_d_display['Num_Unique_Brnd_Names'].max()}")
print(f"Max generic names for a single provider: {provider_summary_d_display['Num_Unique_Gnrc_Names'].max()}")

provider_summary_d_display

In [ ]:
part_d_df['Prscrbr_State_Abrvtn'].unique()

In [ ]:
part_d_df['Prscrbr_Type_Src'].unique()

In [ ]:
print("Part D Data Info:")
part_d_df.info()

### 3.3 Load OIG Exclusion List

In [ ]:
print("Loading OIG Exclusion List...")
oig_df = pd.read_csv(OIG_FILE, low_memory=False)

print(f"OIG Data Shape: {oig_df.shape}")
print(f"\nMemory Usage: {oig_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

oig_df.head()

In [ ]:
print(oig_df.columns)

In [ ]:

print("OIG Data Info:")
oig_df.info()

print("Unique values in key columns:")
print(f"Unique Last Names: {oig_df['LASTNAME'].nunique() if 'LASTNAME' in oig_df.columns else 'N/A'}")
print(f"Unique States: {oig_df['STATE'].nunique() if 'STATE' in oig_df.columns else 'N/A'}")

## 4. Data Quality Assessment

### 4.1 Missing Values Analysis

In [ ]:
def analyze_missing_values(df, df_name):
    print(f"Missing Values Analysis: {df_name}")

    missing_values = ['', ' ', '  ', 'nan', 'NaN', 'NA', 'N/A', 'null', 'NULL', 'None', 'none', '<NA>']
    df_temp = df.replace(missing_values, pd.NA)
    
    missing_count = df_temp.isnull().sum()
    missing_percent = (missing_count / len(df_temp)) * 100
    
    missing_df = pd.DataFrame({
        'Column': df_temp.columns,
        'Missing_Count': missing_count.values,
        'Missing_Percent': missing_percent.values
    })
    
    missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(
        'Missing_Percent', ascending=False
    ).reset_index(drop=True)
    
    if len(missing_df) == 0:
        print("✓ No missing values found!")
    else:
        print(f"\nColumns with missing values: {len(missing_df)}")
        print(missing_df)
        
        if len(missing_df) <= 20:
            fig = px.bar(
                missing_df,
                x='Column',
                y='Missing_Percent',
                title=f'Missing Values in {df_name}',
                labels={'Missing_Percent': 'Missing %'},
                text='Missing_Percent'
            )
            fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
            fig.update_layout(xaxis_tickangle=-45, height=500)
            fig.show()
    
    return missing_df

missing_part_b = analyze_missing_values(part_b_df, 'Medicare Part B 2023')
missing_part_d = analyze_missing_values(part_d_df, 'Medicare Part D 2023')
missing_oig = analyze_missing_values(oig_df, 'OIG Exclusion List')

### 4.2 Duplicate Records Check

In [ ]:
def check_duplicates(df, df_name):
    print(f"\nDuplicate Analysis: {df_name}")

    full_duplicates = df.duplicated().sum()
    print(f"Full duplicate rows: {full_duplicates} ({full_duplicates/len(df)*100:.2f}%)")
    
    return full_duplicates

dup_part_b = check_duplicates(part_b_df, 'Part B')
dup_part_d = check_duplicates(part_d_df, 'Part D')
dup_oig = check_duplicates(oig_df, 'OIG')

## 5. Data Preprocessing & Cleaning


In [ ]:
def basic_data_cleaning(df, df_name):
    
    df_clean = df.copy()
    initial_rows = len(df_clean)
   
    before_dedup = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    after_dedup = len(df_clean)
    removed_dups = before_dedup - after_dedup
    print(f"1. Removed duplicates: {removed_dups:,} rows")
   
    npi_cols = [col for col in df_clean.columns if 'NPI' in col.upper()]
    for npi_col in npi_cols:
        if df_clean[npi_col].dtype == 'object':
            df_clean[npi_col] = df_clean[npi_col].astype(str).str.strip()
            valid_npi = df_clean[npi_col].str.isnumeric() | df_clean[npi_col].isna()
            invalid_count = (~valid_npi).sum()
            if invalid_count > 0:
                df_clean = df_clean[valid_npi]
                print(f"2. Removed invalid NPIs in {npi_col}: {invalid_count:,} rows")
    
    all_nan_rows = df_clean.isnull().all(axis=1).sum()
    if all_nan_rows > 0:
        df_clean = df_clean.dropna(how='all')
        print(f"3. Removed completely empty rows: {all_nan_rows:,}")
    
    final_rows = len(df_clean)
    total_removed = initial_rows - final_rows
    missing_count = df_clean.isnull().sum().sum()
    
    print(f"\nCLEANING SUMMARY:")
    print(f"  Initial rows:     {initial_rows:,}")
    print(f"  Final rows:       {final_rows:,}")
    print(f"  Rows removed:     {total_removed:,} ({total_removed/initial_rows*100:.2f}%)")
    print(f"  Missing values:   {missing_count:,} (PRESERVED for post-aggregation imputation)")
    print("="*80)
    print()
    
    return df_clean


In [ ]:
print("Applying basic data cleaning (no imputation yet)...")
print()

part_b_clean = basic_data_cleaning(part_b_df, "Part B - Provider Services")
part_d_clean = basic_data_cleaning(part_d_df, "Part D - Prescription Drugs")
oig_clean = basic_data_cleaning(oig_df, "OIG - Exclusion List")


print("✓ Basic cleaning complete! Missing values preserved for post-aggregation imputation.")


## 6. Provider-Level Aggregation


### 6.1 Create Provider-Level Aggregations - Part B



In [ ]:
print("Creating provider-level aggregations from Part B data...")

part_b_clean['Weighted_Payment'] = part_b_clean['Tot_Srvcs'] * part_b_clean['Avg_Mdcr_Pymt_Amt']

part_b_agg = part_b_clean.groupby('Rndrng_NPI').agg({
    
    'Rndrng_Prvdr_Last_Org_Name': 'first',
    'Rndrng_Prvdr_First_Name': 'first',
    'Rndrng_Prvdr_State_Abrvtn': 'first',
    'Rndrng_Prvdr_Zip5': 'first',
    'Rndrng_Prvdr_Type': 'first',  
    'Rndrng_Prvdr_Ent_Cd': 'first', 
    'Rndrng_Prvdr_RUCA': 'first',  
    'Tot_Benes': 'mean',  
    'Tot_Srvcs': 'sum',
    'Tot_Bene_Day_Srvcs': 'sum',

    'Avg_Sbmtd_Chrg': ['mean', 'max'],     
    'Avg_Mdcr_Alowd_Amt': ['mean', 'max'],
    'Avg_Mdcr_Pymt_Amt': 'mean',            
    'Avg_Mdcr_Stdzd_Amt': 'mean',         
    'Weighted_Payment': 'sum',              

  
    'HCPCS_Cd': 'nunique',        
    'HCPCS_Drug_Ind': 'nunique',   
    'Place_Of_Srvc': 'nunique',    
}).reset_index()

part_b_agg.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in part_b_agg.columns.values]

column_mapping = {
    'Rndrng_NPI': 'NPI',
    'Rndrng_Prvdr_Last_Org_Name_first': 'Provider_Name',
    'Rndrng_Prvdr_First_Name_first': 'Provider_FirstName',
    'Rndrng_Prvdr_State_Abrvtn_first': 'State',
    'Rndrng_Prvdr_Zip5_first': 'Zipcode',
    'Rndrng_Prvdr_Type_first': 'Provider_Type',
    'Rndrng_Prvdr_Ent_Cd_first': 'Entity_Code',
    'Rndrng_Prvdr_RUCA_first': 'RUCA_Code',

    'Tot_Benes_mean': 'Total_Beneficiaries_B',
    'Tot_Srvcs_sum': 'Total_Services_B',
    'Tot_Bene_Day_Srvcs_sum': 'Total_BeneDay_Services_B',

    'Avg_Sbmtd_Chrg_mean': 'Avg_Submitted_Charge_B',
    'Avg_Sbmtd_Chrg_max': 'Max_Submitted_Charge_B',
    'Avg_Mdcr_Alowd_Amt_mean': 'Avg_Medicare_Allowed_B',
    'Avg_Mdcr_Alowd_Amt_max': 'Max_Medicare_Allowed_B',
    'Avg_Mdcr_Pymt_Amt_mean': 'Avg_Medicare_Payment_B',
    'Avg_Mdcr_Stdzd_Amt_mean': 'Avg_Medicare_Standardized_B',
    'Weighted_Payment_sum': 'Weighted_Avg_Payment_B',  # Will be divided by Total_Services_B below
 
    'HCPCS_Cd_nunique': 'Unique_HCPCS_B',
    'HCPCS_Drug_Ind_nunique': 'Service_Type_Diversity_B',  # 1=one type (Drug/Non-Drug), 2=both
    'Place_Of_Srvc_nunique': 'Unique_Service_Places_B',
}
part_b_agg.rename(columns=column_mapping, inplace=True)


part_b_agg['Weighted_Avg_Payment_B'] = (
    part_b_agg['Weighted_Avg_Payment_B'] / part_b_agg['Total_Services_B']
)

print(f"✓ Part B aggregated to {len(part_b_agg):,} unique providers")
print(f"  Columns: {part_b_agg.shape[1]}")
print(f"  Features: {part_b_agg.shape[1] - 7} numeric + 7 descriptive")
part_b_agg.head()


### 6.2 Create Provider-Level Aggregations - Part D

In [ ]:
print("\nCreating provider-level aggregations from Part D data...")
part_d_agg = part_d_clean.groupby('Prscrbr_NPI').agg({
    'Prscrbr_Last_Org_Name': 'first',
    'Prscrbr_First_Name': 'first',
    'Prscrbr_State_Abrvtn': 'first',
    'Prscrbr_Type': 'first',

    'Tot_Clms': 'sum',          
    'Tot_30day_Fills': 'sum',    
    'Tot_Day_Suply': 'sum',      


    'Tot_Drug_Cst': ['sum', 'mean'],  
    'Brnd_Name': 'nunique',     
    'Gnrc_Name': 'nunique',      
}).reset_index()

part_d_agg.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in part_d_agg.columns.values]

column_mapping = {
    'Prscrbr_NPI': 'NPI',
    'Prscrbr_Last_Org_Name_first': 'Prescriber_Name_D',
    'Prscrbr_First_Name_first': 'Prescriber_FirstName_D',
    'Prscrbr_State_Abrvtn_first': 'Prescriber_State_D',
    'Prscrbr_Type_first': 'Prescriber_Type_D',
    'Tot_Clms_sum': 'Total_Claims_D',
    'Tot_30day_Fills_sum': 'Total_30Day_Fills_D',
    'Tot_Day_Suply_sum': 'Total_Day_Supply_D',
    'Tot_Drug_Cst_sum': 'Total_Drug_Cost_D',
    'Tot_Drug_Cst_mean': 'Avg_Drug_Cost_Per_Claim_D',
    'Brnd_Name_nunique': 'Unique_Brand_Drugs_D',
    'Gnrc_Name_nunique': 'Unique_Generic_Drugs_D',
}
part_d_agg.rename(columns=column_mapping, inplace=True)

print(f"✓ Part D aggregated to {len(part_d_agg):,} unique prescribers")
part_d_agg.head()


### 6.3 Create Fraud Labels from OIG Data

In [ ]:
print("\nCreating fraud labels from OIG exclusion list...")

oig_npi_list = oig_clean['NPI'].dropna().astype(str).str.strip()

oig_npi_list = oig_npi_list[oig_npi_list.str.isnumeric()]
oig_npi_set = set(oig_npi_list.astype(int))
print(f"Total excluded providers in OIG: {len(oig_npi_set):,}")
print("\nMerging Part B and Part D data...")
providers_df = part_b_agg.merge(part_d_agg, on='NPI', how='outer', indicator=True)
print(f"Combined dataset: {len(providers_df):,} providers")
print(f"  - Only in Part B: {(providers_df['_merge'] == 'left_only').sum():,}")
print(f"  - Only in Part D: {(providers_df['_merge'] == 'right_only').sum():,}")
print(f"  - In both datasets: {(providers_df['_merge'] == 'both').sum():,}")

providers_df['Data_Source'] = providers_df['_merge'].map({
    'left_only': 'Part_B_Only',
    'right_only': 'Part_D_Only',
    'both': 'Both'
})
providers_df.drop('_merge', axis=1, inplace=True)

print("\nConsolidating provider information...")
providers_df['Provider_Name'] = providers_df['Provider_Name'].fillna(providers_df['Prescriber_Name_D'])
providers_df['State'] = providers_df['State'].fillna(providers_df['Prescriber_State_D'])
providers_df['Provider_Type'] = providers_df['Provider_Type'].fillna(providers_df['Prescriber_Type_D'])

print("\nCreating fraud labels...")
providers_df['Is_Fraudulent'] = providers_df['NPI'].isin(oig_npi_set).astype(int)

print(f"\nFraud Label Distribution:")
fraud_counts = providers_df['Is_Fraudulent'].value_counts()
print(f"  Non-Fraudulent (0): {fraud_counts.get(0, 0):,}")
print(f"  Fraudulent (1):     {fraud_counts.get(1, 0):,}")
print(f"  Fraud rate:         {providers_df['Is_Fraudulent'].mean()*100:.4f}%")


print(f"\nFraud distribution by data source:")
fraud_by_source = pd.crosstab(providers_df['Data_Source'], providers_df['Is_Fraudulent'], margins=True)
print(fraud_by_source)

print(f"✓ Final merged dataset: {providers_df.shape[0]:,} providers × {providers_df.shape[1]} features")
providers_df.head()

In [ ]:
print("MISSING VALUE ANALYSIS - MERGED DATASET")


missing_data = pd.DataFrame({
    'Column': providers_df.columns,
    'Missing_Count': providers_df.isnull().sum().values,
    'Missing_Percent': (providers_df.isnull().sum().values / len(providers_df) * 100)
})
missing_data = missing_data.sort_values('Missing_Percent', ascending=False)

missing_data_filtered = missing_data[missing_data['Missing_Count'] > 0]

print(f"\nTotal columns: {len(providers_df.columns)}")
print(f"Columns with missing values: {len(missing_data_filtered)}")
print(f"Columns without missing values: {len(providers_df.columns) - len(missing_data_filtered)}")


print(f"\n{'Column':<40} {'Missing Count':<15} {'Missing %':<15}")
for _, row in missing_data_filtered.iterrows():
    print(f"{row['Column']:<40} {int(row['Missing_Count']):<15,} {row['Missing_Percent']:<15.2f}")


import plotly.graph_objects as go


plot_data = missing_data_filtered.head(20) 

fig = go.Figure()

fig.add_trace(go.Bar(
    x=plot_data['Missing_Percent'],
    y=plot_data['Column'],
    orientation='h',
    marker=dict(
        color=plot_data['Missing_Percent'],
        colorscale='Reds',
        showscale=True,
        colorbar=dict(title="Missing %")
    ),
    text=[f"{val:.2f}%" for val in plot_data['Missing_Percent']],
    textposition='auto',
))

fig.update_layout(
    title="Missing Value Percentage by Column (Merged Dataset)",
    xaxis_title="Missing Percentage (%)",
    yaxis_title="Column Name",
    height=max(400, len(plot_data) * 25),
    showlegend=False,
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()


### 6.3.1 Missing Value Analysis

### 6.4 Post-Merge Data Quality & Missing Value Handling



### 6.4.1

In [ ]:
print("VERIFYING POST-MERGE IMPUTATION")

print(f"\nBefore imputation:")
print(f"  Missing values: {providers_df.isnull().sum().sum():,}")

print(f"\nAfter imputation:")
print(f"  Missing values: {providers_df.isnull().sum().sum():,}")

print(f"\nKey columns after imputation:")
print(f"  Entity_Code missing: {providers_df['Entity_Code'].isnull().sum()}")
print(f"  Provider_FirstName missing: {providers_df['Provider_FirstName'].isnull().sum()}")
print(f"  Zipcode missing: {providers_df['Zipcode'].isnull().sum()}")
print(f"  Prescriber_Name_D missing: {providers_df['Prescriber_Name_D'].isnull().sum()}")
print(f"  Prescriber_State_D missing: {providers_df['Prescriber_State_D'].isnull().sum()}")
print(f"  Prescriber_Type_D missing: {providers_df['Prescriber_Type_D'].isnull().sum()}")

print("SECTION 6.4 COMPLETE - FINAL DATASET SUMMARY")

### 6.4.2

In [ ]:

print("SEMANTIC MISSING VALUE IMPUTATION")

def impute_post_merge_missing(df):

    df_imputed = df.copy()
    imputation_log = []

   
    part_d_only = df_imputed.get('Data_Source', pd.Series('', index=df_imputed.index)) == 'Part_D_Only'
    part_b_only = df_imputed.get('Data_Source', pd.Series('', index=df_imputed.index)) == 'Part_B_Only'

  
    if 'Entity_Code' in df_imputed.columns and 'Provider_FirstName' in df_imputed.columns:
        missing_entity = df_imputed['Entity_Code'].isnull()
        if missing_entity.sum() > 0:
          
            df_imputed.loc[missing_entity & part_d_only, 'Entity_Code'] = 'Unknown'

            has_firstname = df_imputed['Provider_FirstName'].notna() & (df_imputed['Provider_FirstName'] != '')
            df_imputed.loc[missing_entity & ~part_d_only & has_firstname, 'Entity_Code'] = 'I'
            df_imputed.loc[missing_entity & ~part_d_only & ~has_firstname, 'Entity_Code'] = 'O'
            imputation_log.append(f"Entity_Code: {missing_entity.sum()} values imputed "
                                  f"('Unknown' for Part D-only; I/O heuristic for Part B providers)")

    if 'Provider_FirstName' in df_imputed.columns:
        missing_fn = df_imputed['Provider_FirstName'].isnull()
        if missing_fn.sum() > 0:
            df_imputed.loc[missing_fn & part_d_only, 'Provider_FirstName'] = 'Unknown'
            df_imputed.loc[missing_fn & ~part_d_only, 'Provider_FirstName'] = ''
            imputation_log.append(f"Provider_FirstName: {missing_fn.sum()} values imputed "
                                  f"('Unknown' for Part D-only; '' for Part B organizations)")

    for state_col in ['State', 'State_B', 'State_D']:
        if state_col in df_imputed.columns:
            missing_state = df_imputed[state_col].isnull()
            if missing_state.sum() > 0:
                df_imputed.loc[missing_state, state_col] = 'Unknown'
                imputation_log.append(f"{state_col}: {missing_state.sum()} values imputed to 'Unknown' "
                                      f"(no state data in either dataset)")

    for type_col in ['Provider_Type', 'Provider_Type_B', 'Provider_Type_D']:
        if type_col in df_imputed.columns:
            missing_type = df_imputed[type_col].isnull()
            if missing_type.sum() > 0:
                df_imputed.loc[missing_type, type_col] = 'Unknown'
                imputation_log.append(f"{type_col}: {missing_type.sum()} values imputed to 'Unknown' "
                                      f"(no specialty data in either dataset)")

    if 'Zipcode' in df_imputed.columns:
        missing_zip = df_imputed['Zipcode'].isnull()
        if missing_zip.sum() > 0:
            df_imputed.loc[missing_zip & part_d_only, 'Zipcode'] = 'Unknown'
            missing_zip_b = df_imputed['Zipcode'].isnull() 
            if missing_zip_b.sum() > 0:
                if 'State' in df_imputed.columns:
                    for state in df_imputed.loc[missing_zip_b, 'State'].unique():
                        state_mask = missing_zip_b & (df_imputed['State'] == state)
                        mode_zip = df_imputed[df_imputed['State'] == state]['Zipcode'].mode()
                        fill_val = mode_zip[0] if len(mode_zip) > 0 else '00000'
                        df_imputed.loc[state_mask, 'Zipcode'] = fill_val
                else:
                    df_imputed.loc[missing_zip_b, 'Zipcode'] = '00000'
            imputation_log.append(f"Zipcode: {missing_zip.sum()} values imputed "
                                  f"('Unknown' for Part D-only; state-based mode for Part B providers)")


    for col in numeric_cols:
        if col not in ['NPI', 'Fraud_Flag']:
            missing_num = df_imputed[col].isnull()
            if missing_num.sum() > 0:
                df_imputed.loc[missing_num, col] = 0
                imputation_log.append(f"{col}: {missing_num.sum()} values imputed to 0 "
                                      f"(no activity / not in source dataset)")


    prescriber_crossfill = {
        'Prescriber_Name_D':      'Provider_Name',
        'Prescriber_FirstName_D': 'Provider_FirstName',
        'Prescriber_State_D':     'State',
        'Prescriber_Type_D':      'Provider_Type',
    }
    for d_col, b_col in prescriber_crossfill.items():
        if d_col in df_imputed.columns:
            missing_val = df_imputed[d_col].isnull()
            if missing_val.sum() > 0:
                if b_col in df_imputed.columns:
                    df_imputed.loc[missing_val, d_col] = df_imputed.loc[missing_val, b_col]
                still_missing = df_imputed[d_col].isnull()
                if still_missing.sum() > 0:
                    df_imputed.loc[still_missing, d_col] = 'Unknown'
                imputation_log.append(f"{d_col}: {missing_val.sum()} values imputed "
                                      f"(cross-filled from {b_col}; 'Unknown' for any remainder)")

    if 'Data_Source' in df_imputed.columns:
        missing_val = df_imputed['Data_Source'].isnull()
        if missing_val.sum() > 0:
            df_imputed.loc[missing_val, 'Data_Source'] = 'Unknown'
            imputation_log.append(f"Data_Source: {missing_val.sum()} values imputed to 'Unknown'")

    print("\nImputation Summary:")
    for log_entry in imputation_log:
        print(f"  • {log_entry}")
    if not imputation_log:
        print("  ✓ No missing values required imputation")

    return df_imputed


providers_df_clean = impute_post_merge_missing(providers_df)

final_missing = providers_df_clean.isnull().sum().sum()

print(f"FINAL VERIFICATION: {final_missing} missing values remaining")

if final_missing == 0:
    print("✓ All missing values successfully imputed!")
else:
    print(f"⚠️  Still have {final_missing} missing values")
    remaining_missing = providers_df_clean.isnull().sum()
    print("\nRemaining missing values:")
    print(remaining_missing[remaining_missing > 0])


### 6.4.3

In [ ]:
providers_df_clean.head(10)

### 6.4.4

In [ ]:
print(f"\nBefore removal: {providers_df_clean.shape}")
print(f"Columns: {len(providers_df_clean.columns)}")

redundant_cols = ['Prescriber_Name_D', 'Prescriber_FirstName_D', 'Prescriber_State_D', 'Prescriber_Type_D']

print(f"\nRemoving redundant prescriber columns (redundant with Provider information):")
for col in redundant_cols:
    if col in providers_df_clean.columns:
        print(f"  ✗ {col}")

providers_df_clean = providers_df_clean.drop(columns=redundant_cols, errors='ignore')


providers_df = providers_df_clean.copy()

print(f"\nAfter removal: {providers_df.shape}")
print(f"Columns: {len(providers_df.columns)}")
print(f"\nRemaining columns:")
print(providers_df.columns.tolist())

print("\n✓ Redundant columns removed successfully!")

In [ ]:
providers_df.head(10)

## 7. Exploratory Data Analysis (EDA)

### 7.1 Feature Engineering

In [ ]:
_n_cols_before = providers_df.shape[1]

print("Section 7.1  Advanced Feature Engineering")
print(f"\nStarting shape  : {providers_df.shape[0]:,} rows × {providers_df.shape[1]:,} columns")
print(f"Missing values  : {providers_df.isnull().sum().sum():,}")


print("\n[1] Payment & charge ratio features ...")

providers_df['Charge_to_Allowed_Ratio'] = np.where(
    providers_df['Avg_Medicare_Allowed_B'] > 0,
    providers_df['Avg_Submitted_Charge_B'] / providers_df['Avg_Medicare_Allowed_B'],
    0
)


providers_df['Payment_to_Allowed_Ratio'] = np.where(
    providers_df['Avg_Medicare_Allowed_B'] > 0,
    providers_df['Avg_Medicare_Payment_B'] / providers_df['Avg_Medicare_Allowed_B'],
    0
)

providers_df['MaxCharge_Ratio'] = np.where(
    providers_df['Avg_Submitted_Charge_B'] > 0,
    providers_df['Max_Submitted_Charge_B'] / providers_df['Avg_Submitted_Charge_B'],
    0
)

providers_df['Allowed_Charge_Gap'] = (
    providers_df['Avg_Submitted_Charge_B'] - providers_df['Avg_Medicare_Allowed_B']
)

providers_df['Payment_Efficiency'] = np.where(
    providers_df['Avg_Submitted_Charge_B'] > 0,
    providers_df['Avg_Medicare_Payment_B'] / providers_df['Avg_Submitted_Charge_B'],
    0
)

providers_df['Standardized_Payment_Ratio'] = np.where(
    providers_df['Avg_Medicare_Standardized_B'] > 0,
    providers_df['Avg_Medicare_Payment_B'] / providers_df['Avg_Medicare_Standardized_B'],
    0
)

print("[2] Service intensity & volume features ...")

providers_df['Services_per_Beneficiary_B'] = np.where(
    providers_df['Total_Beneficiaries_B'] > 0,
    providers_df['Total_Services_B'] / providers_df['Total_Beneficiaries_B'],
    0
)

providers_df['BeneDay_Per_Beneficiary_B'] = np.where(
    providers_df['Total_Beneficiaries_B'] > 0,
    providers_df['Total_BeneDay_Services_B'] / providers_df['Total_Beneficiaries_B'],
    0
)

providers_df['Services_Per_HCPCS'] = np.where(
    providers_df['Unique_HCPCS_B'] > 0,
    providers_df['Total_Services_B'] / providers_df['Unique_HCPCS_B'],
    0
)

providers_df['HCPCS_Diversity'] = np.where(
    providers_df['Total_Services_B'] > 0,
    providers_df['Unique_HCPCS_B'] / providers_df['Total_Services_B'],
    0
)

providers_df['Procedure_Breadth'] = providers_df['Unique_HCPCS_B']


providers_df['Total_Patient_Volume'] = providers_df['Total_Beneficiaries_B']

providers_df['Has_PartB'] = (providers_df['Total_Services_B'] > 0).astype(int)
providers_df['Has_PartD'] = (providers_df['Total_Claims_D']   > 0).astype(int)
providers_df['Has_Both']  = (
    (providers_df['Has_PartB'] == 1) & (providers_df['Has_PartD'] == 1)
).astype(int)


print("[3] Drug cost & prescribing pattern features ...")


providers_df['Drug_Cost_Per_Claim_D'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_Drug_Cost_D'] / providers_df['Total_Claims_D'],
    0
)

providers_df['Avg_Cost_per_Claim_D'] = providers_df['Drug_Cost_Per_Claim_D']


providers_df['Drug_Cost_Per_Day'] = np.where(
    providers_df['Total_Day_Supply_D'] > 0,
    providers_df['Total_Drug_Cost_D'] / providers_df['Total_Day_Supply_D'],
    0
)

providers_df['Fills_Per_Claim_D'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_30Day_Fills_D'] / providers_df['Total_Claims_D'],
    0
)

providers_df['Days_Per_Claim_D'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_Day_Supply_D'] / providers_df['Total_Claims_D'],
    0
)

print("[4] Drug diversity & brand features ...")

providers_df['Brand_Ratio'] = np.where(
    (providers_df['Unique_Brand_Drugs_D'] + providers_df['Unique_Generic_Drugs_D']) > 0,
    providers_df['Unique_Brand_Drugs_D'] /
        (providers_df['Unique_Brand_Drugs_D'] + providers_df['Unique_Generic_Drugs_D']),
    0
)

providers_df['Total_Service_Diversity'] = (
    providers_df['Service_Type_Diversity_B'] +
    providers_df['Unique_Brand_Drugs_D'] +
    providers_df['Unique_Generic_Drugs_D']
)

print("[5] Cross-part B ↔ D features ...")

providers_df['Service_to_Drug_Ratio'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_Services_B'] / providers_df['Total_Claims_D'],
    0
)

print("[6] Log-transform features (log1p, zero-safe) ...")

providers_df['log_Total_Services_B']      = np.log1p(providers_df['Total_Services_B'])
providers_df['log_Total_Beneficiaries_B'] = np.log1p(providers_df['Total_Beneficiaries_B'])
providers_df['log_Total_Claims_D']        = np.log1p(providers_df['Total_Claims_D'])
providers_df['log_Total_Drug_Cost_D']     = np.log1p(providers_df['Total_Drug_Cost_D'])
providers_df['log_Total_Day_Supply_D']    = np.log1p(providers_df['Total_Day_Supply_D'])


print("[7] Within-Provider_Type z-score features ...")

for raw_col, z_col in [
    ('Total_Services_B',      'Z_Total_Services_B'),
    ('Total_Drug_Cost_D',     'Z_Total_Drug_Cost_D'),
    ('Avg_Medicare_Payment_B','Z_Avg_Medicare_Payment_B'),
]:
    _grp  = providers_df.groupby('Provider_Type')[raw_col]
    _mean = _grp.transform('mean')
    _std  = _grp.transform('std').replace(0, np.nan)
    providers_df[z_col] = ((providers_df[raw_col] - _mean) / _std).fillna(0)

print("[8] Binary high-value anomaly flags (z > 1.645 = top-5% within specialty) ...")

providers_df['High_Service_Volume_Flag'] = (
    providers_df['Z_Total_Services_B']       > 1.645).astype(int)
providers_df['High_Drug_Cost_Flag']      = (
    providers_df['Z_Total_Drug_Cost_D']      > 1.645).astype(int)
providers_df['High_Payment_Flag']        = (
    providers_df['Z_Avg_Medicare_Payment_B'] > 1.645).astype(int)


print("[9] Interaction features ...")

providers_df['High_Service_High_Payment'] = (
    providers_df['Services_per_Beneficiary_B'] * providers_df['Avg_Medicare_Payment_B']
)

providers_df['Drug_Intensity'] = (
    providers_df['Drug_Cost_Per_Claim_D'] * providers_df['Brand_Ratio']
)

providers_df['Charge_x_Volume'] = (
    providers_df['Avg_Submitted_Charge_B'] * providers_df['Total_Services_B']
)

_n_new = providers_df.shape[1] - _n_cols_before
print(f"\n{'='*60}")
print(f"Final shape     : {providers_df.shape[0]:,} rows × {providers_df.shape[1]:,} columns")
print(f"Base columns    : {_n_cols_before}")
print(f"New features    : {_n_new}")
print(f"Missing values  : {providers_df.isnull().sum().sum():,}")
print("✓ Feature engineering complete!")

providers_df.head()


#### 7.1.1 Summary Statistics for Engineered Features

In [ ]:
print("7.1.1  Summary Statistics — Engineered Features")


engineered_features = {

    "--- GROUP 1: Payment & Charge Ratios ---": [],
    "Charge_to_Allowed_Ratio"      : "Over-billing rate vs. Medicare allowed (>1 = upcoding risk)",
    "Payment_to_Allowed_Ratio"     : "Medicare payment as fraction of allowed (legitimately ≤1)",
    "MaxCharge_Ratio"              : "Max vs. avg charge — outlier charge spike detector",
    "Allowed_Charge_Gap"           : "Absolute $ over-billing per service line",
    "Payment_Efficiency"           : "Fraction of submitted charge actually paid by Medicare",
    "Standardized_Payment_Ratio"   : "Payment premium above geography-adjusted baseline",

  
    "--- GROUP 2: Service Intensity (Part B) ---": [],
    "Services_per_Beneficiary_B"   : "Average services billed per unique patient",
    "BeneDay_Per_Beneficiary_B"    : "Beneficiary-day services per patient (phantom-billing flag)",
    "Services_Per_HCPCS"           : "Billing code concentration (repetitive-code flag)",
    "HCPCS_Diversity"              : "Code diversity ratio — low = concentrated billing",
    "Procedure_Breadth"            : "Unique HCPCS codes (named alias for Unique_HCPCS_B)",
    "Total_Patient_Volume"         : "Unique patient count (alias for Total_Beneficiaries_B)",
    "Has_PartB"                    : "Binary: provider has Part B activity",
    "Has_PartD"                    : "Binary: provider has Part D activity",
    "Has_Both"                     : "Binary: provider participates in both Part B and D",

    "--- GROUP 3: Drug Cost & Prescribing (Part D) ---": [],
    "Drug_Cost_Per_Claim_D"        : "Weighted avg drug cost per filed claim (sum÷sum)",
    "Avg_Cost_per_Claim_D"         : "Alias for Drug_Cost_Per_Claim_D (backward compat.)",
    "Drug_Cost_Per_Day"            : "Daily drug cost — high = expensive/high-dose prescribing",
    "Fills_Per_Claim_D"            : "30-day fills per claim — >1 flags drug-mill patterns",
    "Days_Per_Claim_D"             : "Avg days supply per claim — very low = frequent refills",

    "--- GROUP 4: Drug Diversity & Brand ---": [],
    "Brand_Ratio"                  : "Fraction of unique drugs that are brand-name (0–1)",
    "Total_Service_Diversity"      : "Breadth of services across Part B + Part D",

    "--- GROUP 5: Cross-Part (B ↔ D) ---": [],
    "Service_to_Drug_Ratio"        : "Part B services / Part D claims (specialty mismatch flag)",

  
    "--- GROUP 6: Log Transforms ---": [],
    "log_Total_Services_B"         : "log(1 + Total_Services_B) — right-skew compression",
    "log_Total_Beneficiaries_B"    : "log(1 + Total_Beneficiaries_B)",
    "log_Total_Claims_D"           : "log(1 + Total_Claims_D)",
    "log_Total_Drug_Cost_D"        : "log(1 + Total_Drug_Cost_D)",
    "log_Total_Day_Supply_D"       : "log(1 + Total_Day_Supply_D)",

    "--- GROUP 7: Within-Provider_Type Z-scores ---": [],
    "Z_Total_Services_B"           : "Peer-relative service volume (vs. same Provider_Type)",
    "Z_Total_Drug_Cost_D"          : "Peer-relative drug spend (vs. same Provider_Type)",
    "Z_Avg_Medicare_Payment_B"     : "Peer-relative payment per service (vs. same Provider_Type)",

    "--- GROUP 8: Binary Anomaly Flags (top-5% within specialty) ---": [],
    "High_Service_Volume_Flag"     : "1 if Z_Total_Services_B > 1.645",
    "High_Drug_Cost_Flag"          : "1 if Z_Total_Drug_Cost_D > 1.645",
    "High_Payment_Flag"            : "1 if Z_Avg_Medicare_Payment_B > 1.645",

    "--- GROUP 9: Interaction Features ---": [],
    "High_Service_High_Payment"    : "Services_per_Beneficiary_B × Avg_Medicare_Payment_B",
    "Drug_Intensity"               : "Drug_Cost_Per_Claim_D × Brand_Ratio",
    "Charge_x_Volume"              : "Avg_Submitted_Charge_B × Total_Services_B (total exposure)",
}
summary_data = []
for key, desc in engineered_features.items():
    if key.startswith("---"):
        print(f"\n{key}")
        print("-" * 80)
        continue
    if key not in providers_df.columns:
        print(f"  ⚠ MISSING: {key}")
        continue
    col = providers_df[key]
    fraud_mean    = providers_df.loc[providers_df['Is_Fraudulent'] == 1, key].mean()
    nonfr_mean    = providers_df.loc[providers_df['Is_Fraudulent'] == 0, key].mean()
    fraud_lift    = (fraud_mean / nonfr_mean) if nonfr_mean != 0 else float('nan')
    print(f"  {key:<32}  min={col.min():>10.3f}  max={col.max():>12.3f}  "
          f"mean={col.mean():>10.3f}  fraud_lift={fraud_lift:>6.2f}x  | {desc[:40]}")
    summary_data.append({
        'Feature'          : key,
        'Min'              : col.min(),
        'Max'              : col.max(),
        'Mean'             : col.mean(),
        'Median'           : col.median(),
        'Fraud Mean'       : fraud_mean,
        'Non-Fraud Mean'   : nonfr_mean,
        'Fraud Lift'       : fraud_lift,
        'Description'      : desc,
    })

summary_df = pd.DataFrame(summary_data)

print(f"✓ Total engineered features tracked : {len(summary_data)}")
print(f"✓ Successfully created              : {sum(1 for d in summary_data if d['Feature'] in providers_df.columns)}")
print(f"\nTop features by Fraud Lift (fraud mean / non-fraud mean):")
top_lift = (summary_df[summary_df['Fraud Lift'].notna() & np.isfinite(summary_df['Fraud Lift'])]
            .nlargest(10, 'Fraud Lift')[['Feature', 'Fraud Mean', 'Non-Fraud Mean', 'Fraud Lift']])
print(top_lift.to_string(index=False))


### 7.2

In [ ]:
fraud_by_state = providers_df[providers_df['Is_Fraudulent'] == 1]['State'].value_counts().head(15)

fraud_rate_by_state = (
    providers_df.groupby('State')['Is_Fraudulent'].agg(['sum', 'count'])
)
fraud_rate_by_state['fraud_rate'] = (fraud_rate_by_state['sum'] / fraud_rate_by_state['count'] * 100)
fraud_rate_by_state = fraud_rate_by_state.sort_values('fraud_rate', ascending=False).head(15)

fig = make_subplots(
    rows=1, cols=1,
    subplot_titles=('Top 15 States by Fraud Rate %',)
)

fig.add_trace(
    go.Bar(x=fraud_rate_by_state['fraud_rate'], y=fraud_rate_by_state.index, 
           orientation='h', name='Fraud Rate %', marker_color='red'),
    row=1, col=1
)

fig.update_layout(height=400, showlegend=False, title_text="Geographic Distribution of Healthcare Fraud")
fig.update_xaxes(title_text="Fraud Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="State", row=1, col=1)
fig.show()

print(f"\nTop states by fraud count:")
print(fraud_by_state)

print(f"\nTop states by fraud rate:")
print(fraud_rate_by_state[['sum', 'count', 'fraud_rate']].head(15))


### 7.3 Specialty Analysis

In [ ]:
print("Analyzing fraud patterns by provider specialty...")

fraud_rate_by_type = (
    providers_df.groupby('Provider_Type')['Is_Fraudulent'].agg(['sum', 'count'])
)
fraud_rate_by_type = fraud_rate_by_type[fraud_rate_by_type['count'] >= 100] 
fraud_rate_by_type['fraud_rate'] = (fraud_rate_by_type['sum'] / fraud_rate_by_type['count'] * 100)
fraud_rate_by_type = fraud_rate_by_type.sort_values('fraud_rate', ascending=False).head(15)

fig = go.Figure()

fig.add_trace(
    go.Bar(y=fraud_rate_by_type.index, x=fraud_rate_by_type['fraud_rate'], orientation='h',
           marker_color='darkred', text=fraud_rate_by_type['fraud_rate'].round(2),
           texttemplate='%{text}%', textposition='outside')
)

fig.update_layout(
    height=600, 
    title_text="Healthcare Fraud Rate by Provider Specialty (min 100 providers)",
    xaxis_title="Fraud Rate (%)",
    yaxis_title="Provider Type",
    showlegend=False
)

fig.show()

print(f"\nTop 15 specialties by fraud rate:")
print(fraud_rate_by_type[['sum', 'count', 'fraud_rate']])


### 7.4 Correlation Analysis

In [ ]:

print("Analyzing feature correlations...")

numeric_features = providers_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_features.remove('NPI') 


corr_matrix = providers_df[numeric_features].corr()


plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


fraud_correlations = corr_matrix['Is_Fraudulent'].sort_values(ascending=False)
print("\nTop 15 features correlated with fraud:")
print(fraud_correlations.head(16)[1:]) 

print("\nBottom 10 features (negative correlation with fraud):")
print(fraud_correlations.tail(10))

## 8. Statistical Feature Selection


### 8.1

In [ ]:

export_dir = OUTPUT_DIR / 'datasets'
export_dir.mkdir(parents=True, exist_ok=True)

X_train_current = X_train_reduced.copy() if 'X_train_reduced' in globals() else X_train.copy()
X_test_current = X_test_reduced.copy() if 'X_test_reduced' in globals() else X_test.copy()
y_train_current = y_train.copy()
y_test_current = y_test.copy()


train_export = X_train_current.copy()
train_export['Is_Fraudulent'] = y_train_current.values

test_export = X_test_current.copy()
test_export['Is_Fraudulent'] = y_test_current.values

X_train_current.to_csv(export_dir / 'X_train_reduced_all fea_SMOTE.csv', index=True, index_label='original_index')
X_test_current.to_csv(export_dir / 'X_test_reduced_all fea_SMOTE.csv', index=True, index_label='original_index')
y_train_current.to_csv(export_dir / 'y_train_all fea_SMOTE.csv', index=True, index_label='original_index', header=True)
y_test_current.to_csv(export_dir / 'y_test_all fea_SMOTE.csv', index=True, index_label='original_index', header=True)

train_export.to_csv(export_dir / 'train_set_reduced_all fea_SMOTE.csv', index=True, index_label='original_index')
test_export.to_csv(export_dir / 'test_set_reduced_all fea_SMOTE.csv', index=True, index_label='original_index')

pd.Series(X_train_current.columns, name='feature').to_csv(
    export_dir / 'selected_features_statistical_all fea_SMOTE.csv', index=False
)

print("✓ Train/test sets exported")
print(f"Export folder: {export_dir}")
print(f"Train features: {X_train_current.shape}")
print(f"Test features:  {X_test_current.shape}")
print(f"Train labels:   {y_train_current.shape}")
print(f"Test labels:    {y_test_current.shape}")
print("\nFiles created:")
for file_name in [
    'X_train_reduced_all fea_SMOTE.csv',
    'X_test_reduced_all fea_SMOTE.csv',
    'y_train_all fea_SMOTE.csv',
    'y_test_all fea_SMOTE.csv',
    'train_set_reduced_all fea_SMOTE.csv',
    'test_set_reduced_all fea_SMOTE.csv',
    'selected_features_statistical_all fea_SMOTE.csv'
]:
    print(f"  - {file_name}")

### 8.3

In [ ]:
export_dir = OUTPUT_DIR / 'datasets'

X_train_reduced = pd.read_csv(export_dir / 'X_train_reduced_all fea_SMOTE.csv', index_col='original_index')
X_test_reduced = pd.read_csv(export_dir / 'X_test_reduced_all fea_SMOTE.csv', index_col='original_index')

y_train = pd.read_csv(export_dir / 'y_train_all fea_SMOTE.csv', index_col='original_index').iloc[:, 0]
y_test = pd.read_csv(export_dir / 'y_test_all fea_SMOTE.csv', index_col='original_index').iloc[:, 0]

selected_features_statistical = pd.read_csv(
    export_dir / 'selected_features_statistical_all fea_SMOTE.csv'
)['feature'].tolist()

train_export = pd.read_csv(export_dir / 'train_set_reduced_all fea_SMOTE.csv', index_col='original_index')
test_export = pd.read_csv(export_dir / 'test_set_reduced_all fea_SMOTE.csv', index_col='original_index')

print("✓ Dumped files loaded successfully")
print(f"Folder: {export_dir}")
print(f"X_train_reduced: {X_train_reduced.shape}")
print(f"X_test_reduced:  {X_test_reduced.shape}")
print(f"y_train:         {y_train.shape}")
print(f"y_test:          {y_test.shape}")
print(f"Selected features: {len(selected_features_statistical)}")

print("\nFirst 5 selected features:")
print(selected_features_statistical[:5])

### 8.4

In [ ]:
from sklearn.feature_selection import VarianceThreshold

print("SECTION 8: STATISTICAL FEATURE SELECTION")
print("NOTE: Correlation-based removal is DISABLED in this notebook (ML_Fraud_Detection_2).")
print("      All features, including highly correlated ones, are kept for modelling.")

if 'providers_df' not in vars() and 'providers_df' not in globals():
    print("\n⚠️  providers_df not found — detected reload mode (Section 8.3).")
    print("   Pre-processed datasets are already loaded; skipping full pipeline.")
    print("CHI-SQUARE TEST on loaded X_train_reduced (informational)")
    non_negative_features = X_train_reduced.columns[(X_train_reduced >= 0).all()].tolist()
    if len(non_negative_features) > 0:
        chi_scores, p_values = chi2(X_train_reduced[non_negative_features], y_train)
        chi_scores_df = pd.DataFrame({
            'Feature': non_negative_features,
            'Chi2_Score': chi_scores,
            'P_Value': p_values
        }).sort_values('Chi2_Score', ascending=False)
        print("\nTop 10 features by Chi-Square score:")
        print(chi_scores_df.head(10).to_string(index=False))
        significant_features = chi_scores_df[chi_scores_df['P_Value'] < 0.05]['Feature'].tolist()
        print(f"\nStatistically significant features (p < 0.05): {len(significant_features)}")
    else:
        print("⚠️ No non-negative features available for chi-square test")
        significant_features = []
        chi_scores_df = None

    to_drop_corr = []
    low_var_features = []

    feature_selection_results = {
        'original_features': X_train_reduced.columns.tolist(),
        'selected_features': selected_features_statistical,
        'removed_by_correlation': to_drop_corr,
        'removed_by_variance': low_var_features,
        'chi_scores': chi_scores_df
    }

    print("FEATURE SELECTION SUMMARY  (reload mode — data loaded from CSV)")
    print(f"\n✓ X_train_reduced: {X_train_reduced.shape}")
    print(f"✓ X_test_reduced:  {X_test_reduced.shape}")
    print(f"✓ y_train:         {y_train.shape}")
    print(f"✓ y_test:          {y_test.shape}")
    print(f"✓ selected_features_statistical: {len(selected_features_statistical)} features")
    print("\n✓ STATISTICAL FEATURE SELECTION COMPLETE (reload mode)")

else:
    columns_to_drop_for_selection = ['NPI', 'Is_Fraudulent', 'Provider_Name', 'Provider_FirstName',
                                       'State', 'Zipcode', 'Provider_Type', 'Entity_Code', 'Data_Source']
    existing_cols_to_drop_selection = [col for col in columns_to_drop_for_selection if col in providers_df.columns]

    print(f"\n  • Dropping {len(existing_cols_to_drop_selection)} non-predictive columns: {existing_cols_to_drop_selection}")

    X_all = providers_df.drop(columns=existing_cols_to_drop_selection)
    y_all = providers_df['Is_Fraudulent']

    print(f"\n✓ Total predictive features: {X_all.shape[1]}")
    print(f"✓ Total samples: {X_all.shape[0]:,}")
    print(f"✓ Fraud cases: {y_all.sum():,} ({y_all.sum()/len(y_all)*100:.2f}%)")

    print("STEP 0: TRAIN-TEST SPLIT (80-20)")


    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all,
        test_size=0.2,
        random_state=42,
        stratify=y_all
    )

    print(f"\nTrain set: {X_train.shape[0]:,} samples ({y_train.sum():,} fraud cases, {y_train.sum()/len(y_train)*100:.2f}%)")
    print(f"Test set:  {X_test.shape[0]:,} samples ({y_test.sum():,} fraud cases, {y_test.sum()/len(y_test)*100:.2f}%)")

    to_drop_corr = []          
    X_train_reduced = X_train.copy()
    X_test_reduced  = X_test.copy()
  
    print("STEP 1: CORRELATION-BASED FEATURE REMOVAL — SKIPPED")

    print(f"  → Keeping all {X_train_reduced.shape[1]} features (no correlation filter applied).")

    print("STEP 2: VARIANCE THRESHOLD (on training data)")


    var_threshold = 0.01
    selector_var = VarianceThreshold(threshold=var_threshold)
    selector_var.fit(X_train_reduced)

    feature_variances = X_train_reduced.var()
    low_var_features = feature_variances[feature_variances < var_threshold].index.tolist()

    print(f"\nVariance threshold: {var_threshold}")
    print(f"Low variance features to remove: {len(low_var_features)}")
    if low_var_features:
        print("\nLow variance features:")
        for feat in low_var_features:
            print(f"  • {feat} (variance: {feature_variances[feat]:.6f})")

    X_train_reduced = pd.DataFrame(
        selector_var.transform(X_train_reduced),
        columns=X_train_reduced.columns[selector_var.get_support()],
        index=X_train_reduced.index
    )
    X_test_reduced = pd.DataFrame(
        selector_var.transform(X_test_reduced),
        columns=X_test_reduced.columns[selector_var.get_support()],
        index=X_test_reduced.index
    )
    print(f"\n✓ Features after variance threshold: {X_train_reduced.shape[1]}")

    print("STEP 3: CHI-SQUARE TEST (on training data)")


    non_negative_features = X_train_reduced.columns[(X_train_reduced >= 0).all()].tolist()

    if len(non_negative_features) > 0:
        print(f"\nNon-negative features for chi-square test: {len(non_negative_features)}")

        chi_scores, p_values = chi2(X_train_reduced[non_negative_features], y_train)

        chi_scores_df = pd.DataFrame({
            'Feature': non_negative_features,
            'Chi2_Score': chi_scores,
            'P_Value': p_values
        }).sort_values('Chi2_Score', ascending=False)

        print("\nTop 10 features by Chi-Square score:")
        print(chi_scores_df.head(10).to_string(index=False))

        significant_features = chi_scores_df[chi_scores_df['P_Value'] < 0.05]['Feature'].tolist()
        print(f"\nStatistically significant features (p < 0.05): {len(significant_features)}")
    else:
        print("\n⚠️ No non-negative features available for chi-square test")
        significant_features = []
        chi_scores_df = None

    print("FEATURE SELECTION SUMMARY  (ML_Fraud_Detection_2 — no correlation removal)")
  

    print(f"\n1. Original features:             {X_all.shape[1]}")
    print(f"2. After correlation removal:      {X_all.shape[1]}  ← SKIPPED (all kept)")
    print(f"3. After variance threshold:       {X_train_reduced.shape[1]}")
    print(f"4. Statistically significant (χ²): {len(significant_features)}")

    print(f"\n✓ Final features: {X_train_reduced.shape[1]}")
    print(f"✓ Features removed (variance only): {X_all.shape[1] - X_train_reduced.shape[1]}")
    print(f"✓ Reduction: {(1 - X_train_reduced.shape[1]/X_all.shape[1])*100:.1f}%")

    selected_features_statistical = X_train_reduced.columns.tolist()
    print(f"\nSelected features list ({len(selected_features_statistical)} features):")
    for i, feat in enumerate(selected_features_statistical, 1):
        print(f"  {i}. {feat}")
    feature_selection_results = {
        'original_features': X_all.columns.tolist(),
        'selected_features': selected_features_statistical,
        'removed_by_correlation': to_drop_corr,  
        'removed_by_variance': low_var_features,
        'chi_scores': chi_scores_df
    }

    print("\n✓ STATISTICAL FEATURE SELECTION COMPLETE")
    print("✓ Feature selection applied on training data only — no data leakage!")


## 9. Data Scaling & Class Balancing


In [ ]:
print("DATA PREPARATION SUMMARY")


print("\n📊 Using pre-split data from Section 8.1:")
print(f"   • Train-test split already performed (80-20 stratified)")
print(f"   • Feature selection applied on training data only")
print(f"   • No data leakage - test data never influenced feature selection")

print(f"\n✓ Training set shape: {X_train_reduced.shape}")
print(f"✓ Test set shape: {X_test_reduced.shape}")

print(f"\n✓ Training fraud rate: {y_train.mean()*100:.4f}% ({y_train.sum():,} / {len(y_train):,})")
print(f"✓ Test fraud rate: {y_test.mean()*100:.4f}% ({y_test.sum():,} / {len(y_test):,})")

print(f"\n✓ Selected features for modeling: {len(selected_features_statistical)}")
print("\nFeature list:")
for i, feat in enumerate(selected_features_statistical, 1):
    print(f"  {i:2d}. {feat}")

X_train = X_train_reduced.copy()
X_test = X_test_reduced.copy()
feature_names = X_train.columns.tolist()

print(f"\n✓ Ready for modeling with {len(feature_names)} features")


### 9.2 Handle Class Imbalance

In [ ]:
print("Handling class imbalance with SMOTE...")

print(f"Before SMOTE:")
print(f"  Class 0 (Non-Fraud): {(y_train == 0).sum()}")
print(f"  Class 1 (Fraud): {(y_train == 1).sum()}")
print(f"  Imbalance ratio: {(y_train == 0).sum() / (y_train == 1).sum():.2f}:1")

smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"  Class 0 (Non-Fraud): {(y_train_balanced == 0).sum()}")
print(f"  Class 1 (Fraud): {(y_train_balanced == 1).sum()}")
print(f"  Imbalance ratio: {(y_train_balanced == 0).sum() / (y_train_balanced == 1).sum():.2f}:1")

print(f"\nNew training set shape: {X_train_balanced.shape}")

### 9.3 Feature Scaling

In [ ]:
print("Scaling features using RobustScaler...")

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Training data scaled: {X_train_scaled.shape}")
print(f"✓ Test data scaled: {X_test_scaled.shape}")

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names)

print("\nData is ready for model training!")

### 9.4 Model Evaluation Function

Define the evaluation function before training models.

In [ ]:
def evaluate_model(y_true, y_pred, y_pred_proba, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_pred_proba)
    mcc = matthews_corrcoef(y_true, y_pred)

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"MCC:       {mcc:.4f}")
    
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(cm)
    
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred))
    
    return {
        'model': model_name,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'mcc': mcc
    }

print("✓ Model evaluation function defined")

## 10. Baseline Models

In [ ]:

results_list = []


print("MODEL TRAINING AND EVALUATION")
print("\nInitialized results tracking...")
print(f"Test set size: {len(y_test):,} samples")
print(f"Test set fraud cases: {y_test.sum():,} ({y_test.sum()/len(y_test)*100:.2f}%)")

y_test.value_counts()

### 10.1 Logistic Regression

In [ ]:
print("Training Logistic Regression Model...")

lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train_balanced)

y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_results = evaluate_model(y_test, y_pred_lr, y_pred_proba_lr, "Logistic Regression")
results_list.append(lr_results)

print("\n✓ Logistic Regression training complete!")

## 14. Model Optimization

### 14.1 Hyperparameter Tuning

In [ ]:
# Random Forest 

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import numpy as np


print("Section 14.1a  Random Forest — Hyperopt v2")



if 'X_train_orig_scaled' not in dir():
    X_train_orig_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=feature_names, index=X_train.index
    )


X_tr_inner, X_val_inner, y_tr_inner, y_val_inner = train_test_split(
    X_train_orig_scaled, y_train,
    test_size=0.20, random_state=RANDOM_STATE + 1, stratify=y_train
)

_smote_inner = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_tr_smote, y_tr_smote = _smote_inner.fit_resample(X_tr_inner, y_tr_inner)

print(f"Inner SMOTE-train : {X_tr_smote.shape[0]:,} samples  "
      f"(fraud {y_tr_smote.mean():.2%})")
print(f"Inner val (real)  : {X_val_inner.shape[0]:,} samples  "
      f"(fraud {y_val_inner.mean():.4%})")

rf_space = {
    'n_estimators'     : hp.choice('rf_n_est',   [200, 300, 400, 500]),
    'max_depth'        : hp.choice('rf_depth',   [4, 5, 6, 8, 10]),
    'min_samples_leaf' : hp.choice('rf_leaf',    [2, 4, 6, 8]),
    'min_samples_split': hp.choice('rf_split',   [5, 8, 10, 15, 20]),
    'max_features'     : hp.choice('rf_feats',   ['sqrt', 'log2']),
    'max_samples'      : hp.uniform('rf_samp',   0.55, 0.85),
}

def rf_objective(params):
    clf = RandomForestClassifier(
        **params, random_state=RANDOM_STATE, n_jobs=-1
    )
    clf.fit(X_tr_smote, y_tr_smote)
    proba = clf.predict_proba(X_val_inner)[:, 1]
    return {'loss': -roc_auc_score(y_val_inner, proba), 'status': STATUS_OK}

trials_rf = Trials()
print("\nRunning Hyperopt — 50 evaluations…")
best_rf_idx = fmin(
    fn=rf_objective,
    space=rf_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials_rf,
    rstate=np.random.default_rng(RANDOM_STATE),
    verbose=False,
)

best_rf_params = space_eval(rf_space, best_rf_idx)
print(f"\nBest RF params  : {best_rf_params}")
print(f"Best val ROC-AUC: {-min(trials_rf.losses()):.4f}")

best_rf_full = RandomForestClassifier(
    **best_rf_params, random_state=RANDOM_STATE, n_jobs=-1
)
best_rf_full.fit(X_train_scaled, y_train_balanced)

calibrated_rf_final = CalibratedClassifierCV(
    best_rf_full, cv='prefit', method='isotonic'
)
calibrated_rf_final.fit(X_val_inner, y_val_inner)
best_rf_model = calibrated_rf_final

val_proba_rf = best_rf_model.predict_proba(X_val_inner)[:, 1]
fprs_rf, tprs_rf, roc_thr_rf = roc_curve(y_val_inner, val_proba_rf)
youden_rf   = tprs_rf - fprs_rf
best_idx_rf = int(np.argmax(youden_rf))
best_thr_rf = float(roc_thr_rf[best_idx_rf])
print(f"\nYouden threshold: {best_thr_rf:.4f}  "
      f"TPR={tprs_rf[best_idx_rf]:.4f}  FPR={fprs_rf[best_idx_rf]:.4f}")

rf_tuned_proba  = best_rf_model.predict_proba(X_test_scaled)[:, 1]
rf_tuned_pred   = (rf_tuned_proba >= best_thr_rf).astype(int)
rf_tuned_results = evaluate_model(
    y_test, rf_tuned_pred, rf_tuned_proba, 'Random Forest (Tuned)'
)
results_list = [r for r in results_list if r['model'] != 'Random Forest (Tuned)']
results_list.append(rf_tuned_results)
print("\n✓ Random Forest Hyperopt v2 complete!")


In [ ]:
# XGBoost

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve, roc_auc_score
import numpy as np

print("Section 14.1b  XGBoost — Hyperopt v2")


if xgb is None:
    print("⚠  XGBoost not available — skipping.")
else:
    print(f"\nSMOTE inner-train : {X_tr_smote.shape[0]:,} samples")
    print(f"Original-dist val : {X_val_inner.shape[0]:,} samples  "
          f"(fraud {y_val_inner.mean():.4%})")

    xgb_space = {
        'n_estimators'    : hp.choice('xgb_n_est',    [100, 150, 200, 300, 400]),
        'max_depth'       : hp.choice('xgb_depth',    [3, 4, 5, 6]),
        'learning_rate'   : hp.loguniform('xgb_lr',   np.log(0.01), np.log(0.2)),
        'subsample'       : hp.uniform('xgb_sub',     0.70, 1.00),
        'colsample_bytree': hp.uniform('xgb_col',     0.60, 1.00),
        'min_child_weight': hp.choice('xgb_mcw',      [1, 3, 5, 10]),
        'gamma'           : hp.uniform('xgb_gamma',   0.10, 0.60),
        'max_delta_step'  : hp.choice('xgb_mds',      [1, 3, 5]),
        'reg_alpha'       : hp.loguniform('xgb_alpha', np.log(0.01), np.log(1.0)),
        'reg_lambda'      : hp.uniform('xgb_lambda',  1.0, 5.0),
    }

    def xgb_objective(params):
        clf = xgb.XGBClassifier(
            **params,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method='hist',
            eval_metric='logloss',
            verbosity=0,
        )
        clf.fit(X_tr_smote, y_tr_smote)
        proba = clf.predict_proba(X_val_inner)[:, 1]
        return {'loss': -roc_auc_score(y_val_inner, proba), 'status': STATUS_OK}

    trials_xgb = Trials()
    print("\nRunning Hyperopt — 50 evaluations…")
    best_xgb_idx = fmin(
        fn=xgb_objective,
        space=xgb_space,
        algo=tpe.suggest,
        max_evals=50,
        trials=trials_xgb,
        rstate=np.random.default_rng(RANDOM_STATE),
        verbose=False,
    )

    best_xgb_params = space_eval(xgb_space, best_xgb_idx)
    print(f"\nBest XGB params  : {best_xgb_params}")
    print(f"Best val ROC-AUC : {-min(trials_xgb.losses()):.4f}")

    best_xgb_full = xgb.XGBClassifier(
        **best_xgb_params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method='hist',
        eval_metric='logloss',
        verbosity=0,
    )
    best_xgb_full.fit(X_train_scaled, y_train_balanced)

    calibrated_xgb_final = CalibratedClassifierCV(
        best_xgb_full, cv='prefit', method='isotonic'
    )
    calibrated_xgb_final.fit(X_val_inner, y_val_inner)
    best_xgb_model = calibrated_xgb_final

    val_proba_xgb = best_xgb_model.predict_proba(X_val_inner)[:, 1]
    fprs_xgb, tprs_xgb, roc_thr_xgb = roc_curve(y_val_inner, val_proba_xgb)
    youden_xgb   = tprs_xgb - fprs_xgb
    best_idx_xgb = int(np.argmax(youden_xgb))
    best_thr_xgb = float(roc_thr_xgb[best_idx_xgb])
    print(f"\nYouden threshold: {best_thr_xgb:.4f}  "
          f"TPR={tprs_xgb[best_idx_xgb]:.4f}  FPR={fprs_xgb[best_idx_xgb]:.4f}")

    xgb_tuned_proba  = best_xgb_model.predict_proba(X_test_scaled)[:, 1]
    xgb_tuned_pred   = (xgb_tuned_proba >= best_thr_xgb).astype(int)
    xgb_tuned_results = evaluate_model(
        y_test, xgb_tuned_pred, xgb_tuned_proba, 'XGBoost (Tuned)'
    )
    results_list = [r for r in results_list if r['model'] != 'XGBoost (Tuned)']
    results_list.append(xgb_tuned_results)
    print("\n✓ XGBoost Hyperopt v2 complete!")


In [ ]:
# LightGBM

try:
    import lightgbm as lgb
    from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.metrics import roc_curve, roc_auc_score
    import numpy as np

    print("Section 14.1c  LightGBM — Hyperopt v2")
    print(f"\nSMOTE inner-train : {X_tr_smote.shape[0]:,} samples")
    print(f"Original-dist val : {X_val_inner.shape[0]:,} samples  "
          f"(fraud {y_val_inner.mean():.4%})")

    lgb_space = {
        'n_estimators'     : hp.choice('lgb_n_est',   [100, 200, 300, 400]),
        'max_depth'        : hp.choice('lgb_depth',   [4, 6, 8, 10, 12]),
        'learning_rate'    : hp.loguniform('lgb_lr',  np.log(0.01), np.log(0.3)),
        'num_leaves'       : hp.choice('lgb_leaves',  [31, 63, 127]),
        'subsample'        : hp.uniform('lgb_sub',    0.70, 1.00),
        'colsample_bytree' : hp.uniform('lgb_col',    0.70, 1.00),
        'min_child_samples': hp.choice('lgb_mcs',     [10, 20, 50, 100]),
        'lambda_l1'        : hp.loguniform('lgb_l1',  np.log(0.001), np.log(1.0)),
        'lambda_l2'        : hp.loguniform('lgb_l2',  np.log(0.01),  np.log(2.0)),
    }

    def lgb_objective(params):
        clf = lgb.LGBMClassifier(
            **params,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            bagging_freq=1,
            verbosity=-1,
        )
        clf.fit(X_tr_smote, y_tr_smote)
        proba = clf.predict_proba(X_val_inner)[:, 1]
        return {'loss': -roc_auc_score(y_val_inner, proba), 'status': STATUS_OK}

    trials_lgb = Trials()
    print("\nRunning Hyperopt — 50 evaluations…")
    best_lgb_idx = fmin(
        fn=lgb_objective,
        space=lgb_space,
        algo=tpe.suggest,
        max_evals=50,
        trials=trials_lgb,
        rstate=np.random.default_rng(RANDOM_STATE),
        verbose=False,
    )

    best_lgb_params = space_eval(lgb_space, best_lgb_idx)
    print(f"\nBest LGB params  : {best_lgb_params}")
    print(f"Best val ROC-AUC : {-min(trials_lgb.losses()):.4f}")

    best_lgb_full = lgb.LGBMClassifier(
        **best_lgb_params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        bagging_freq=1,
        verbosity=-1,
    )
    best_lgb_full.fit(X_train_scaled, y_train_balanced)

    calibrated_lgb_final = CalibratedClassifierCV(
        best_lgb_full, cv='prefit', method='isotonic'
    )
    calibrated_lgb_final.fit(X_val_inner, y_val_inner)
    best_lgb_model = calibrated_lgb_final

    val_proba_lgb = best_lgb_model.predict_proba(X_val_inner)[:, 1]
    fprs_lgb, tprs_lgb, roc_thr_lgb = roc_curve(y_val_inner, val_proba_lgb)
    youden_lgb   = tprs_lgb - fprs_lgb
    best_idx_lgb = int(np.argmax(youden_lgb))
    best_thr_lgb = float(roc_thr_lgb[best_idx_lgb])
    print(f"\nYouden threshold: {best_thr_lgb:.4f}  "
          f"TPR={tprs_lgb[best_idx_lgb]:.4f}  FPR={fprs_lgb[best_idx_lgb]:.4f}")

    lgb_tuned_proba  = best_lgb_model.predict_proba(X_test_scaled)[:, 1]
    lgb_tuned_pred   = (lgb_tuned_proba >= best_thr_lgb).astype(int)
    lgb_tuned_results = evaluate_model(
        y_test, lgb_tuned_pred, lgb_tuned_proba, 'LightGBM (Tuned)'
    )
    results_list = [r for r in results_list if r['model'] != 'LightGBM (Tuned)']
    results_list.append(lgb_tuned_results)
    print("\n✓ LightGBM Hyperopt v2 complete!")

except ImportError:
    print("⚠  LightGBM not available — skipping Section 14.1c.")


In [ ]:

# HistGradientBoosting 

from sklearn.ensemble import HistGradientBoostingClassifier
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve, roc_auc_score
import numpy as np

print("Section 14.1d  HistGradientBoosting — Hyperopt v2")
print(f"\nSMOTE inner-train : {X_tr_smote.shape[0]:,} samples")
print(f"Original-dist val : {X_val_inner.shape[0]:,} samples  "
      f"(fraud {y_val_inner.mean():.4%})")

hgb_space = {
    'max_iter'         : hp.choice('hgb_iters',   [200, 300, 400, 500]),
    'max_depth'        : hp.choice('hgb_depth',   [4, 6, 8, 10, 12]),
    'learning_rate'    : hp.loguniform('hgb_lr',  np.log(0.03), np.log(0.5)),
    'max_leaf_nodes'   : hp.choice('hgb_leaves',  [31, 63, 127]),
    'min_samples_leaf' : hp.choice('hgb_msl',     [20, 50, 100, 200]),
    'l2_regularization': hp.loguniform('hgb_l2',  np.log(0.1), np.log(10.0)),
}

def hgb_objective(params):
    clf = HistGradientBoostingClassifier(
        **params,
        random_state=RANDOM_STATE,
        early_stopping=False,
        verbose=0,
    )
    clf.fit(X_tr_smote, y_tr_smote)
    proba = clf.predict_proba(X_val_inner)[:, 1]
    return {'loss': -roc_auc_score(y_val_inner, proba), 'status': STATUS_OK}

trials_hgb = Trials()
print("\nRunning Hyperopt — 50 evaluations…")
best_hgb_idx = fmin(
    fn=hgb_objective,
    space=hgb_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials_hgb,
    rstate=np.random.default_rng(RANDOM_STATE),
    verbose=False,
)

best_hgb_params = space_eval(hgb_space, best_hgb_idx)
print(f"\nBest HGB params  : {best_hgb_params}")
print(f"Best val ROC-AUC : {-min(trials_hgb.losses()):.4f}")

best_hgb_full = HistGradientBoostingClassifier(
    **best_hgb_params,
    random_state=RANDOM_STATE,
    early_stopping=False,
    verbose=0,
)
best_hgb_full.fit(X_train_scaled, y_train_balanced)

calibrated_hgb_final = CalibratedClassifierCV(
    best_hgb_full, cv='prefit', method='isotonic'
)
calibrated_hgb_final.fit(X_val_inner, y_val_inner)
best_hgb_model = calibrated_hgb_final

val_proba_hgb = best_hgb_model.predict_proba(X_val_inner)[:, 1]
fprs_hgb, tprs_hgb, roc_thr_hgb = roc_curve(y_val_inner, val_proba_hgb)
youden_hgb   = tprs_hgb - fprs_hgb
best_idx_hgb = int(np.argmax(youden_hgb))
best_thr_hgb = float(roc_thr_hgb[best_idx_hgb])
print(f"\nYouden threshold: {best_thr_hgb:.4f}  "
      f"TPR={tprs_hgb[best_idx_hgb]:.4f}  FPR={fprs_hgb[best_idx_hgb]:.4f}")

hgb_tuned_proba  = best_hgb_model.predict_proba(X_test_scaled)[:, 1]
hgb_tuned_pred   = (hgb_tuned_proba >= best_thr_hgb).astype(int)
hgb_tuned_results = evaluate_model(
    y_test, hgb_tuned_pred, hgb_tuned_proba, 'HistGradBoost (Tuned)'
)
results_list = [r for r in results_list if r['model'] != 'HistGradBoost (Tuned)']
results_list.append(hgb_tuned_results)
print("\n✓ HistGradientBoosting Hyperopt v2 complete!")


In [ ]:
# SVM 
from sklearn.svm import LinearSVC
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import numpy as np

print("Section 14.1e  SVM (LinearSVC) — Hyperopt v2")

if 'X_tr_smote' not in globals():
    if 'X_train_orig_scaled' not in globals():
        X_train_orig_scaled = pd.DataFrame(
            scaler.transform(X_train), columns=feature_names, index=X_train.index
        )
    X_tr_orig, X_val_inner, y_tr_orig, y_val_inner = train_test_split(
        X_train_orig_scaled, y_train,
        test_size=0.20,
        random_state=RANDOM_STATE + 1,
        stratify=y_train
    )
    _smote_inner = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    X_tr_smote, y_tr_smote = _smote_inner.fit_resample(X_tr_orig, y_tr_orig)
    print("  → Recreated X_tr_smote / X_val_inner from X_train_orig_scaled")

print(f"\nSMOTE inner-train : {X_tr_smote.shape[0]:,} samples")
print(f"Original-dist val : {X_val_inner.shape[0]:,} samples  "
      f"(fraud {y_val_inner.mean():.4%})")


MAX_SVM_TRAIN = 200_000
if X_tr_smote.shape[0] > MAX_SVM_TRAIN:
    _svm_idx = np.random.RandomState(RANDOM_STATE).choice(
        X_tr_smote.shape[0], MAX_SVM_TRAIN, replace=False
    )
    X_tr_svm_sub = X_tr_smote.iloc[_svm_idx] if hasattr(X_tr_smote, 'iloc') else X_tr_smote[_svm_idx]
    y_tr_svm_sub = y_tr_smote.iloc[_svm_idx] if hasattr(y_tr_smote, 'iloc') else y_tr_smote[_svm_idx]
    print(f"  → subsampled to {MAX_SVM_TRAIN:,} rows for LinearSVC")
else:
    X_tr_svm_sub = X_tr_smote
    y_tr_svm_sub = y_tr_smote

svm_space = {
    'C': hp.loguniform('C', np.log(0.001), np.log(1000.0)),
}

def svm_objective(params):
    clf = LinearSVC(
        C=params['C'],
        loss='squared_hinge',
        max_iter=5000,
        random_state=RANDOM_STATE,
    )
    clf.fit(X_tr_svm_sub, y_tr_svm_sub)
    scores = clf.decision_function(X_val_inner)
    return {'loss': -roc_auc_score(y_val_inner, scores), 'status': STATUS_OK}

trials_svm = Trials()
print("\nRunning Hyperopt — 25 evaluations…")
best_svm_idx = fmin(
    fn=svm_objective,
    space=svm_space,
    algo=tpe.suggest,
    max_evals=25,
    trials=trials_svm,
    rstate=np.random.default_rng(RANDOM_STATE),
    verbose=False,
)

best_svm_params = space_eval(svm_space, best_svm_idx)
best_svm_C = float(best_svm_params['C'])
print(f"\nBest SVM C       : {best_svm_C:.6f}")
print(f"Best val ROC-AUC : {-min(trials_svm.losses()):.4f}")

best_svm_full = LinearSVC(
    C=best_svm_C,
    loss='squared_hinge',
    max_iter=5000,
    random_state=RANDOM_STATE,
)
best_svm_full.fit(X_train_scaled, y_train_balanced)

calibrated_svm_final = CalibratedClassifierCV(
    best_svm_full, cv='prefit', method='sigmoid'
)
calibrated_svm_final.fit(X_val_inner, y_val_inner)
best_svm_model = calibrated_svm_final

val_proba_svm = best_svm_model.predict_proba(X_val_inner)[:, 1]
fprs_svm, tprs_svm, roc_thr_svm = roc_curve(y_val_inner, val_proba_svm)
youden_svm   = tprs_svm - fprs_svm
best_idx_svm = int(np.argmax(youden_svm))
best_thr_svm = float(roc_thr_svm[best_idx_svm])
print(f"\nYouden threshold: {best_thr_svm:.4f}  "
      f"TPR={tprs_svm[best_idx_svm]:.4f}  FPR={fprs_svm[best_idx_svm]:.4f}")

svm_tuned_proba  = best_svm_model.predict_proba(X_test_scaled)[:, 1]
svm_tuned_pred   = (svm_tuned_proba >= best_thr_svm).astype(int)
svm_tuned_results = evaluate_model(
    y_test, svm_tuned_pred, svm_tuned_proba, 'SVM (Tuned)'
)
results_list = [r for r in results_list if r['model'] != 'SVM (Tuned)']
results_list.append(svm_tuned_results)
print("\n✓ SVM Hyperopt v2 complete!")


In [ ]:
# MLPClassifier

from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier

if 'y_train_balanced' not in globals():
    print("⚠️  Prerequisites not found — running minimal setup from saved CSVs…")
    import numpy as np
    import pandas as pd
    from pathlib import Path
    from imblearn.over_sampling import SMOTE
    from sklearn.preprocessing import RobustScaler
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        roc_auc_score, confusion_matrix, classification_report, matthews_corrcoef
    )

    RANDOM_STATE = 42
    OUTPUT_DIR   = Path('/Users/roya/Desktop/Thesis/Thesis_Project/ML')
    export_dir   = OUTPUT_DIR / 'datasets'

    X_train_reduced = pd.read_csv(
        export_dir / 'X_train_reduced_all fea_SMOTE.csv', index_col='original_index')
    X_test_reduced  = pd.read_csv(
        export_dir / 'X_test_reduced_all fea_SMOTE.csv',  index_col='original_index')
    y_train = pd.read_csv(
        export_dir / 'y_train_all fea_SMOTE.csv', index_col='original_index').iloc[:, 0]
    y_test  = pd.read_csv(
        export_dir / 'y_test_all fea_SMOTE.csv',  index_col='original_index').iloc[:, 0]

    X_train       = X_train_reduced.copy()
    X_test        = X_test_reduced.copy()
    feature_names = X_train.columns.tolist()

    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

    scaler         = RobustScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train_balanced), columns=feature_names)
    X_test_scaled  = pd.DataFrame(
        scaler.transform(X_test), columns=feature_names)

    if 'results_list' not in globals():
        results_list = []

    if 'evaluate_model' not in globals():
        def evaluate_model(y_true, y_pred, y_pred_proba, model_name):
            accuracy  = accuracy_score(y_true, y_pred)
            precision = precision_score(y_true, y_pred)
            recall    = recall_score(y_true, y_pred)
            f1        = f1_score(y_true, y_pred)
            roc_auc   = roc_auc_score(y_true, y_pred_proba)
            mcc       = matthews_corrcoef(y_true, y_pred)
            print(f"Accuracy:  {accuracy:.4f}\nPrecision: {precision:.4f}\n"
                  f"Recall:    {recall:.4f}\nF1-Score:  {f1:.4f}\n"
                  f"ROC-AUC:   {roc_auc:.4f}\nMCC:       {mcc:.4f}")
            cm = confusion_matrix(y_true, y_pred)
            print(f"\nConfusion Matrix:\n{cm}")
            print(f"\nClassification Report:\n{classification_report(y_true, y_pred)}")
            return {'model': model_name, 'accuracy': accuracy, 'precision': precision,
                    'recall': recall, 'f1': f1, 'roc_auc': roc_auc, 'mcc': mcc}

    print("✓ Minimal prerequisites initialized from saved CSV files")

print("Section 14.1f  Hyperparameter Tuning: Neural Network (MLP)")

print(f"\nBalanced training set (SMOTE-ENN) : {len(y_train_balanced):,} samples")
print(f"  Class 0 : {int((y_train_balanced==0).sum()):,}  |  "
      f"Class 1 : {int((y_train_balanced==1).sum()):,}")
print(f"Full training samples : {len(y_train):,}  (original, pre-SMOTE)")
print(f"Fraud cases           : {int(y_train.sum()):,}  ({y_train.mean():.4%})")

if "X_train_orig_scaled" not in globals():
    X_train_orig_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=feature_names, index=X_train.index
    )
X_tr_mlp, X_val_mlp, y_tr_mlp_orig, y_val_mlp = train_test_split(
    X_train_orig_scaled, y_train,
    test_size=0.20, random_state=RANDOM_STATE + 5, stratify=y_train
)
print(f"\nTrain subset : {len(y_tr_mlp_orig):,} | Val subset : {len(y_val_mlp):,}")
print(f"Val fraud    : {int(y_val_mlp.sum()):,}")

_MLP_CV_CAP = 20_000
if len(y_train_balanced) > _MLP_CV_CAP:
    _rng = np.random.RandomState(RANDOM_STATE)
    _idx = _rng.choice(len(y_train_balanced), size=_MLP_CV_CAP, replace=False)
    if hasattr(X_train_scaled, 'iloc'):
        X_mlp_cv = X_train_scaled.iloc[_idx]
        y_mlp_cv = y_train_balanced.iloc[_idx]
    else:
        X_mlp_cv = X_train_scaled[_idx]
        y_mlp_cv = y_train_balanced[_idx]
else:
    X_mlp_cv = X_train_scaled
    y_mlp_cv = y_train_balanced

print(f"\n⚡ CV search subsample  : {len(y_mlp_cv):,} rows  "
      f"(fraud: {int((y_mlp_cv==1).sum()):,}, {(y_mlp_cv==1).mean():.4%})")

cv_mlp = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

mlp_param_dist = {
    "hidden_layer_sizes" : [(128, 64, 32),
                            (256, 128, 64, 32),
                            (256, 128, 64),
                            (512, 256, 128, 64),
                            (128, 64)],
    "alpha"              : [0.0001, 0.001, 0.01],
    "learning_rate_init" : [0.001, 0.0005, 0.0001],
    "batch_size"         : [128, 256, 512],
}

mlp_search = RandomizedSearchCV(
    estimator=MLPClassifier(
        activation='relu',
        solver='adam',
        learning_rate='adaptive',
        max_iter=300,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=15,
        verbose=False,
    ),
    param_distributions=mlp_param_dist,
    n_iter=10,                 
    scoring="roc_auc",
    cv=cv_mlp,
    n_jobs=1,                   
    verbose=1,
    random_state=RANDOM_STATE,
    refit=True,
)

print(f"\nTuning MLP (10 iterations × 3 folds = 30 fits on {len(y_mlp_cv):,} rows)…")
mlp_search.fit(X_mlp_cv, y_mlp_cv)
best_mlp_params = mlp_search.best_params_
print(f"\nBest MLP params  : {best_mlp_params}")
print(f"Best CV ROC-AUC  : {mlp_search.best_score_:.4f}")

best_mlp_uncalib = mlp_search.best_estimator_
calibrated_mlp = CalibratedClassifierCV(best_mlp_uncalib, cv="prefit", method="sigmoid")
calibrated_mlp.fit(X_val_mlp, y_val_mlp)

val_proba_mlp    = calibrated_mlp.predict_proba(X_val_mlp)[:, 1]
mlp_fraud_probas = val_proba_mlp[y_val_mlp == 1]
print(f"\nCalibrated fraud probabilities — min: {mlp_fraud_probas.min():.4f}  "
      f"max: {mlp_fraud_probas.max():.4f}  mean: {mlp_fraud_probas.mean():.4f}")

fprs_mlp, tprs_mlp, roc_thr_mlp = roc_curve(y_val_mlp, val_proba_mlp)
youden_mlp   = tprs_mlp - fprs_mlp
best_idx_mlp = int(np.argmax(youden_mlp))
best_thr_mlp = float(roc_thr_mlp[best_idx_mlp])
print(f"\nYouden's J threshold : {best_thr_mlp:.4f}")
print(f"  Val TPR (recall)   : {tprs_mlp[best_idx_mlp]:.4f}")
print(f"  Val FPR            : {fprs_mlp[best_idx_mlp]:.4f}")

print(f"\nRefitting best params on full balanced set ({len(y_train_balanced):,} rows)…")
best_mlp_full = MLPClassifier(
    **best_mlp_params,
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    max_iter=500,
    random_state=RANDOM_STATE,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=20,
    verbose=False,
)
best_mlp_full.fit(X_train_scaled, y_train_balanced)

calibrated_mlp_final = CalibratedClassifierCV(best_mlp_full, cv="prefit", method="sigmoid")
calibrated_mlp_final.fit(X_val_mlp, y_val_mlp)
best_mlp_model = calibrated_mlp_final

mlp_tuned_proba  = best_mlp_model.predict_proba(X_test_scaled)[:, 1]
mlp_tuned_pred   = (mlp_tuned_proba >= best_thr_mlp).astype(int)

mlp_tuned_results = evaluate_model(
    y_test, mlp_tuned_pred, mlp_tuned_proba, "MLP (Tuned)"
)
results_list.append(mlp_tuned_results)

print("\n✓ MLP tuning complete!")


In [ ]:
# GraphSAGE

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from sklearn.neighbors import kneighbors_graph, NearestNeighbors
from sklearn.linear_model import LogisticRegression as _CalLR
from sklearn.metrics import roc_curve

class _Adam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        self.params = [p for p in params if p.requires_grad]
        self.lr = lr; self.b1, self.b2 = betas
        self.eps = eps; self.wd = weight_decay; self.t = 0
        self.m = [torch.zeros_like(p.data) for p in self.params]
        self.v = [torch.zeros_like(p.data) for p in self.params]

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_()
                p.grad.zero_()

    def step(self):
        self.t += 1
        bc1, bc2 = 1 - self.b1 ** self.t, 1 - self.b2 ** self.t
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            g = p.grad.data
            if self.wd:
                g = g.add(p.data, alpha=self.wd)
            self.m[i].mul_(self.b1).add_(g, alpha=1 - self.b1)
            self.v[i].mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
            m_hat = self.m[i] / bc1
            v_hat = self.v[i] / bc2
            p.data.addcdiv_(m_hat, v_hat.sqrt().add_(self.eps), value=-self.lr)


print("Section 14.1g  Graph Neural Network (GraphSAGE + HAN)")

print(f"  PyTorch {torch.__version__}  (pure-torch, no torch_geometric needed)")


_dev = (torch.device("mps")  if torch.backends.mps.is_available() else
        torch.device("cuda") if torch.cuda.is_available() else
        torch.device("cpu"))
print(f"  Device : {_dev}  (scatter aggregation runs on CPU — MPS-safe)")

def _np32(s): return (s.values if hasattr(s,'values') else np.array(s)).astype('float32')
def _np64(s): return (s.values if hasattr(s,'values') else np.array(s)).astype('int64')

_X_tr_full = _np32(X_train_scaled);   _y_tr_full = _np64(y_train_balanced)
_X_te = _np32(X_test_scaled);         _y_te = _np64(y_test)
_Xo   = _np32(X_train_orig_scaled);   _yo  = _np64(y_train)  # original unbalanced

print(f"\nFull balanced training set : {len(_y_tr_full):,}")

_GNN_CAP = 30_000
if len(_y_tr_full) > _GNN_CAP:
    _rng_gnn = np.random.RandomState(RANDOM_STATE + 3)
    _half = _GNN_CAP // 2
    _idx0 = _rng_gnn.choice(np.where(_y_tr_full == 0)[0], size=_half, replace=False)
    _idx1 = _rng_gnn.choice(np.where(_y_tr_full == 1)[0], size=_half, replace=False)
    _gnn_idx = np.concatenate([_idx0, _idx1])
    _rng_gnn.shuffle(_gnn_idx)
    _X_tr = _X_tr_full[_gnn_idx]
    _y_tr = _y_tr_full[_gnn_idx]
    print(f"  Subsampled to {len(_y_tr):,} balanced rows for graph construction")
else:
    _X_tr = _X_tr_full
    _y_tr = _y_tr_full

N_TR, N_FEAT = _X_tr.shape

print(f"\nTraining nodes : {N_TR:,}  "
      f"(fraud: {(_y_tr==1).sum():,} | non-fraud: {(_y_tr==0).sum():,})")
print(f"Test nodes     : {len(_y_te):,}  (fraud: {(_y_te==1).sum()})")

K_NEIGH = 10
print(f"\nBuilding k-NN similarity graph (k={K_NEIGH}) on {N_TR:,} training nodes ...")

A_knn = kneighbors_graph(_X_tr, n_neighbors=K_NEIGH,
                          mode='connectivity', include_self=False, n_jobs=-1)
A_sym = A_knn + A_knn.T
A_sym.data[:] = 1.0
A_coo = A_sym.tocoo()

_ei_src = torch.tensor(A_coo.row.astype('int64'), dtype=torch.long)
_ei_dst = torch.tensor(A_coo.col.astype('int64'), dtype=torch.long)
_deg    = torch.bincount(_ei_src, minlength=N_TR).float().clamp(min=1)
print(f"  Undirected edges : {_ei_src.size(0):,}")

def _mean_agg(H, src, dst, deg):
    dev   = H.device
    H_c   = H.cpu()
    src_c = src.cpu();  dst_c = dst.cpu()
    out   = torch.zeros(deg.size(0), H_c.size(1), dtype=H_c.dtype)
    out.scatter_add_(0, src_c.unsqueeze(1).expand(-1, H_c.size(1)), H_c[dst_c])
    return (out / deg.cpu().unsqueeze(1)).to(dev)

class GraphSAGE(nn.Module):
    def __init__(self, in_ch, hid_ch=64, out_ch=2, drop=0.3):
        super().__init__()
        self.fc1s = nn.Linear(in_ch,  hid_ch)
        self.fc1n = nn.Linear(in_ch,  hid_ch, bias=False)
        self.fc2s = nn.Linear(hid_ch, out_ch)
        self.fc2n = nn.Linear(hid_ch, out_ch, bias=False)
        self.drop = drop

    def forward(self, x, src, dst, deg):
        h1 = F.relu(self.fc1s(x) + self.fc1n(_mean_agg(x, src, dst, deg)))
        h1 = F.dropout(h1, p=self.drop, training=self.training)
        return self.fc2s(h1) + self.fc2n(_mean_agg(h1, src, dst, deg))

sage = GraphSAGE(N_FEAT, hid_ch=64, out_ch=2).to(_dev)
print(f"\nGraphSAGE : {N_FEAT} -> SAGEConv(64) -> ReLU/Drop(0.3) -> SAGEConv(2)")
print(f"  Parameters : {sum(p.numel() for p in sage.parameters()):,}")

X_tr_t = torch.from_numpy(_X_tr).to(_dev)
y_tr_t = torch.from_numpy(_y_tr).to(_dev)
src_t  = _ei_src;  dst_t = _ei_dst;  deg_t = _deg   

_w1  = float((_y_tr==0).sum()) / max(float((_y_tr==1).sum()), 1)
crit = nn.CrossEntropyLoss(weight=torch.tensor([1.0, _w1]).to(_dev))
opt  = _Adam(sage.parameters(), lr=0.005, weight_decay=1e-4)

MAX_EP = 200; PAT = 25; PAT_LR = 10
best_loss = float('inf'); wait = 0; wait_lr = 0; best_lr_loss = float('inf'); best_st = None

print(f"\nTraining ({MAX_EP} max epochs, patience={PAT}) ...")
sage.train()
for ep in range(1, MAX_EP + 1):
    opt.zero_grad()
    loss = crit(sage(X_tr_t, src_t, dst_t, deg_t), y_tr_t)
    loss.backward()
    opt.step()

    lv = loss.item()
    if lv < best_lr_loss: best_lr_loss = lv; wait_lr = 0
    else:
        wait_lr += 1
        if wait_lr >= PAT_LR: opt.lr *= 0.5; wait_lr = 0; best_lr_loss = lv

    if lv < best_loss:
        best_loss = lv; wait = 0
        best_st = {k: v.clone() for k, v in sage.state_dict().items()}
    else:
        wait += 1
        if wait >= PAT:
            print(f"  Early stop at epoch {ep}  (best loss={best_loss:.5f})"); break
    if ep % 50 == 0 or ep == 1:
        print(f"  Epoch {ep:3d}  loss={lv:.5f}  lr={opt.lr:.5f}")

sage.load_state_dict(best_st)
print(f"  Best training loss = {best_loss:.5f}")

sage.eval()
with torch.no_grad():
    h1_tr_t = F.relu(sage.fc1s(X_tr_t) + sage.fc1n(_mean_agg(X_tr_t, src_t, dst_t, deg_t)))
h1_tr_np = h1_tr_t.cpu().numpy()
print(f"\nPrecomputed layer-1 train embeddings : {h1_tr_np.shape}")


nn_idx = NearestNeighbors(n_neighbors=K_NEIGH, n_jobs=-1).fit(_X_tr)

def _infer(X_query, batch=4096):
    n = len(X_query);  _, nbr = nn_idx.kneighbors(X_query)
    out = []
    with torch.no_grad():
        for s in range(0, n, batch):
            e   = min(s + batch, n)
            xq  = torch.from_numpy(X_query[s:e]).to(_dev)
            idx = nbr[s:e]
            xn  = torch.from_numpy(_X_tr[idx].mean(axis=1).astype('float32')).to(_dev)
            h1  = F.relu(sage.fc1s(xq) + sage.fc1n(xn))
            h1n = torch.from_numpy(h1_tr_np[idx].mean(axis=1).astype('float32')).to(_dev)
            h2  = sage.fc2s(h1) + sage.fc2n(h1n)
            out.append(F.softmax(h2, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)


_rng_v      = np.random.RandomState(RANDOM_STATE + 7)
_fraud_idx  = np.where(_yo == 1)[0]
_nf_idx     = np.where(_yo == 0)[0]
_sel_nf     = _rng_v.choice(_nf_idx, size=len(_fraud_idx), replace=False)
_bal_idx    = np.concatenate([_fraud_idx, _sel_nf])
_Xbal       = _Xo[_bal_idx]
_ybal       = _yo[_bal_idx]
print(f"\nBalanced val set : {len(_ybal):,} nodes  "
      f"(fraud: {(_ybal==1).sum()}  |  non-fraud: {(_ybal==0).sum()})")

print(f"Inductive inference -- balanced val ({len(_ybal):,} nodes) ...")
_val_raw_bal = _infer(_Xbal)
print(f"Inductive inference -- test set ({len(_y_te):,} nodes) ...")
_test_raw    = _infer(_X_te)

_fpr_v, _tpr_v, _thr_v = roc_curve(_ybal, _val_raw_bal)
_bj      = int(np.argmax(_tpr_v - _fpr_v))
_opt_thr = float(_thr_v[_bj])

if not np.isfinite(_opt_thr):
    _opt_thr = float(np.percentile(_test_raw, 90))
    print(f"  (degenerate ROC -- using 90th-percentile fallback threshold)")

print(f"\nYouden's J threshold (balanced val) : {_opt_thr:.4f}")
print(f"  Val TPR (recall) : {_tpr_v[_bj]:.4f}")
print(f"  Val FPR          : {_fpr_v[_bj]:.4f}")

_fp = _test_raw[_y_te == 1]
print(f"\nRaw fraud probs (test) -- "
      f"min:{_fp.min():.4f}  max:{_fp.max():.4f}  mean:{_fp.mean():.4f}")

results_list[:] = [r for r in results_list if r.get('Model') != 'GraphSAGE (GNN)']
_pred = (_test_raw >= _opt_thr).astype(int)
sage_results = evaluate_model(y_test, _pred, _test_raw, "GraphSAGE (GNN)")
results_list.append(sage_results)
print("\n  GraphSAGE evaluation complete!")


### Section 14.1h  FNN — Deep Neural Network (DNN)

In [ ]:
# DNN
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_curve

print("Section 14.1h  FNN: Deep Neural Network (DNN)")

_dnn_dev = (torch.device("mps")  if torch.backends.mps.is_available() else
            torch.device("cuda") if torch.cuda.is_available() else
            torch.device("cpu"))
print(f"  Device : {_dnn_dev}")

def _to_f32(s): return (s.values if hasattr(s, 'values') else np.array(s)).astype('float32')
def _to_i64(s): return (s.values if hasattr(s, 'values') else np.array(s)).astype('int64')

_X_dnn_tr = torch.from_numpy(_to_f32(X_train_scaled))    # CPU
_y_dnn_tr = torch.from_numpy(_to_i64(y_train_balanced))  # CPU
_X_dnn_te = torch.from_numpy(_to_f32(X_test_scaled))     # CPU
_n_dnn    = _X_dnn_tr.shape[1]
_w1_dnn   = float((_y_dnn_tr == 0).sum()) / max(float((_y_dnn_tr == 1).sum()), 1)

print(f"\nTrain : {_X_dnn_tr.shape[0]:,}  |  Features : {_n_dnn}")
print(f"Test  : {_X_dnn_te.shape[0]:,}  |  Fraud class weight : {_w1_dnn:.2f}")
class _DNN(nn.Module):
    def __init__(self, in_ch, hidden=(512, 256, 128, 64), drop=0.3):
        super().__init__()
        layers = []
        prev = in_ch
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(drop)]
            prev = h
        layers.append(nn.Linear(prev, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

dnn_model = _DNN(_n_dnn).to(_dnn_dev)
_dnn_params = sum(p.numel() for p in dnn_model.parameters())
print(f"\nDNN : {_n_dnn} -> 512 -> 256 -> 128 -> 64 -> 2  (BN + ReLU + Drop each layer)")
print(f"  Parameters : {_dnn_params:,}")

class _PlateauLR:
    def __init__(self, opt, factor=0.5, patience=5, min_lr=1e-6):
        self.opt = opt; self.factor = factor
        self.patience = patience; self.min_lr = min_lr
        self.best = float('inf'); self.wait = 0
    def step(self, metric):
        if metric < self.best:
            self.best = metric; self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.opt.lr = max(self.opt.lr * self.factor, self.min_lr)
                self.wait = 0

_dnn_crit  = nn.CrossEntropyLoss(weight=torch.tensor([1.0, _w1_dnn]).to(_dnn_dev))
_dnn_opt   = _Adam(dnn_model.parameters(), lr=1e-3, weight_decay=1e-4)
_dnn_sched = _PlateauLR(_dnn_opt, factor=0.5, patience=5)

_DNN_BS  = 2048
_DNN_EP  = 100
_DNN_PAT = 15
_dnn_best_loss = float('inf'); _dnn_wait = 0; _dnn_best_st = None
_n_dnn_tr = _X_dnn_tr.shape[0]

print(f"\nTraining ({_DNN_EP} max epochs, batch={_DNN_BS:,}, patience={_DNN_PAT}) ...")
dnn_model.train()
for _ep in range(1, _DNN_EP + 1):
    _perm     = torch.randperm(_n_dnn_tr)
    _ep_loss  = 0.0;  _steps = 0
    for _s in range(0, _n_dnn_tr, _DNN_BS):
        _idx = _perm[_s:_s + _DNN_BS]
        _xb  = _X_dnn_tr[_idx].to(_dnn_dev)
        _yb  = _y_dnn_tr[_idx].to(_dnn_dev)
        _dnn_opt.zero_grad()
        _l = _dnn_crit(dnn_model(_xb), _yb)
        _l.backward()
        _dnn_opt.step()
        _ep_loss += _l.item();  _steps += 1
    _ep_loss /= _steps
    _dnn_sched.step(_ep_loss)
    if _ep_loss < _dnn_best_loss:
        _dnn_best_loss = _ep_loss;  _dnn_wait = 0
        _dnn_best_st = {k: v.clone() for k, v in dnn_model.state_dict().items()}
    else:
        _dnn_wait += 1
        if _dnn_wait >= _DNN_PAT:
            print(f"  Early stop at epoch {_ep}  (best loss={_dnn_best_loss:.5f})")
            break
    if _ep % 20 == 0 or _ep == 1:
        print(f"  Epoch {_ep:3d}  loss={_ep_loss:.5f}  lr={_dnn_opt.lr:.6f}")

dnn_model.load_state_dict(_dnn_best_st)
print(f"  Best training loss = {_dnn_best_loss:.5f}")
_rng_dval = np.random.RandomState(RANDOM_STATE + 9)
_yo_dnn   = _to_i64(y_train)
_Xo_dnn   = _to_f32(X_train_orig_scaled)
_fi_d     = np.where(_yo_dnn == 1)[0]
_nf_d     = np.where(_yo_dnn == 0)[0]
_sel_d    = _rng_dval.choice(_nf_d, size=len(_fi_d), replace=False)
_bval_d   = np.concatenate([_fi_d, _sel_d])
_Xbval_d  = torch.from_numpy(_Xo_dnn[_bval_d]).to(_dnn_dev)
_ybval_d  = _yo_dnn[_bval_d]

dnn_model.eval()
with torch.no_grad():
    _dnn_val_proba = F.softmax(dnn_model(_Xbval_d), dim=1)[:, 1].cpu().numpy()

_fpr_d, _tpr_d, _thr_d = roc_curve(_ybval_d, _dnn_val_proba)
_bj_d    = int(np.argmax(_tpr_d - _fpr_d))
_thr_dnn = float(_thr_d[_bj_d])
print(f"\nYouden's J threshold : {_thr_dnn:.4f}")
print(f"  Val TPR (recall)   : {_tpr_d[_bj_d]:.4f}")
print(f"  Val FPR            : {_fpr_d[_bj_d]:.4f}")

_dnn_proba_parts = []
with torch.no_grad():
    for _s in range(0, len(_X_dnn_te), 4096):
        _xb = _X_dnn_te[_s:_s + 4096].to(_dnn_dev)
        _dnn_proba_parts.append(F.softmax(dnn_model(_xb), dim=1)[:, 1].cpu().numpy())
_dnn_proba = np.concatenate(_dnn_proba_parts)
_dnn_pred  = (_dnn_proba >= _thr_dnn).astype(int)

dnn_results = evaluate_model(y_test, _dnn_pred, _dnn_proba, "DNN (FNN)")
results_list.append(dnn_results)
print("\n  DNN evaluation complete!")


### FNN 

In [ ]:
# FNN 

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_curve

print("Section 14.1i  FNN: Tabular Residual Network (TabResNet)")
_res_dev = (torch.device("mps")  if torch.backends.mps.is_available() else
            torch.device("cuda") if torch.cuda.is_available() else
            torch.device("cpu"))
print(f"  Device : {_res_dev}")

def _to_f32r(s): return (s.values if hasattr(s, 'values') else np.array(s)).astype('float32')
def _to_i64r(s): return (s.values if hasattr(s, 'values') else np.array(s)).astype('int64')

_X_res_tr = torch.from_numpy(_to_f32r(X_train_scaled))    
_y_res_tr = torch.from_numpy(_to_i64r(y_train_balanced))  
_X_res_te = torch.from_numpy(_to_f32r(X_test_scaled))     
_n_res    = _X_res_tr.shape[1]
_w1_res   = float((_y_res_tr == 0).sum()) / max(float((_y_res_tr == 1).sum()), 1)

print(f"\nTrain : {_X_res_tr.shape[0]:,}  |  Features : {_n_res}")
print(f"Test  : {_X_res_te.shape[0]:,}  |  Fraud class weight : {_w1_res:.2f}")

class _ResBlock(nn.Module):
    def __init__(self, dim, drop=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )

    def forward(self, x):
        return F.relu(self.block(x) + x)   

class _TabResNet(nn.Module):
    def __init__(self, in_ch, hidden=256, n_blocks=4, drop=0.3):
        super().__init__()
        self.proj   = nn.Sequential(
            nn.Linear(in_ch, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
        )
        self.blocks = nn.Sequential(*[_ResBlock(hidden, drop) for _ in range(n_blocks)])
        self.head   = nn.Linear(hidden, 2)

    def forward(self, x):
        return self.head(self.blocks(self.proj(x)))

tab_res = _TabResNet(_n_res, hidden=256, n_blocks=4).to(_res_dev)
_res_params = sum(p.numel() for p in tab_res.parameters())
print(f"\nTabResNet : {_n_res} -> proj(256) -> 4 x ResBlock(256) -> 2")
print(f"  Parameters : {_res_params:,}")

class _PlateauLR:
    def __init__(self, opt, factor=0.5, patience=5, min_lr=1e-6):
        self.opt = opt; self.factor = factor
        self.patience = patience; self.min_lr = min_lr
        self.best = float('inf'); self.wait = 0
    def step(self, metric):
        if metric < self.best:
            self.best = metric; self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.opt.lr = max(self.opt.lr * self.factor, self.min_lr)
                self.wait = 0

_res_crit  = nn.CrossEntropyLoss(weight=torch.tensor([1.0, _w1_res]).to(_res_dev))
_res_opt   = _Adam(tab_res.parameters(), lr=1e-3, weight_decay=1e-4)
_res_sched = _PlateauLR(_res_opt, factor=0.5, patience=5)

_RES_BS  = 2048
_RES_EP  = 100
_RES_PAT = 15
_res_best_loss = float('inf');  _res_wait = 0;  _res_best_st = None
_n_res_tr = _X_res_tr.shape[0]

print(f"\nTraining ({_RES_EP} max epochs, batch={_RES_BS:,}, patience={_RES_PAT}) ...")
tab_res.train()
for _ep in range(1, _RES_EP + 1):
    _perm     = torch.randperm(_n_res_tr)
    _ep_loss  = 0.0;  _steps = 0
    for _s in range(0, _n_res_tr, _RES_BS):
        _idx = _perm[_s:_s + _RES_BS]
        _xb  = _X_res_tr[_idx].to(_res_dev)
        _yb  = _y_res_tr[_idx].to(_res_dev)
        _res_opt.zero_grad()
        _l = _res_crit(tab_res(_xb), _yb)
        _l.backward()
        _res_opt.step()
        _ep_loss += _l.item();  _steps += 1
    _ep_loss /= _steps
    _res_sched.step(_ep_loss)
    if _ep_loss < _res_best_loss:
        _res_best_loss = _ep_loss;  _res_wait = 0
        _res_best_st = {k: v.clone() for k, v in tab_res.state_dict().items()}
    else:
        _res_wait += 1
        if _res_wait >= _RES_PAT:
            print(f"  Early stop at epoch {_ep}  (best loss={_res_best_loss:.5f})")
            break
    if _ep % 20 == 0 or _ep == 1:
        print(f"  Epoch {_ep:3d}  loss={_ep_loss:.5f}  lr={_res_opt.lr:.6f}")

tab_res.load_state_dict(_res_best_st)
print(f"  Best training loss = {_res_best_loss:.5f}")
_rng_rval = np.random.RandomState(RANDOM_STATE + 11)
_yo_res   = _to_i64r(y_train)
_Xo_res   = _to_f32r(X_train_orig_scaled)
_fi_r     = np.where(_yo_res == 1)[0]
_nf_r     = np.where(_yo_res == 0)[0]
_sel_r    = _rng_rval.choice(_nf_r, size=len(_fi_r), replace=False)
_bval_r   = np.concatenate([_fi_r, _sel_r])
_Xbval_r  = torch.from_numpy(_Xo_res[_bval_r]).to(_res_dev)
_ybval_r  = _yo_res[_bval_r]

tab_res.eval()
with torch.no_grad():
    _res_val_proba = F.softmax(tab_res(_Xbval_r), dim=1)[:, 1].cpu().numpy()

_fpr_r, _tpr_r, _thr_r = roc_curve(_ybval_r, _res_val_proba)
_bj_r    = int(np.argmax(_tpr_r - _fpr_r))
_thr_res = float(_thr_r[_bj_r])
print(f"\nYouden's J threshold : {_thr_res:.4f}")
print(f"  Val TPR (recall)   : {_tpr_r[_bj_r]:.4f}")
print(f"  Val FPR            : {_fpr_r[_bj_r]:.4f}")

_res_proba_parts = []
with torch.no_grad():
    for _s in range(0, len(_X_res_te), 4096):
        _xb = _X_res_te[_s:_s + 4096].to(_res_dev)
        _res_proba_parts.append(F.softmax(tab_res(_xb), dim=1)[:, 1].cpu().numpy())
_res_proba = np.concatenate(_res_proba_parts)
_res_pred  = (_res_proba >= _thr_res).astype(int)

res_results = evaluate_model(y_test, _res_pred, _res_proba, "TabResNet (FNN)")
results_list.append(res_results)
print("\n  TabResNet evaluation complete!")
